# TwinPast matched sub-second refinement

Run on an AutoDL RTX 4090 instance with **Restart Kernel → Run All**. On a fresh
container, plan for around 40 GiB of free space on `/root/autodl-tmp`. The notebook
sets up its environment and downloads the pinned 7B model and verified clips as
needed. It can also reuse valid assets from earlier runs on the same container.
It screens twelve new Historical State Recall candidates against prior footage and
evaluates up to eight accepted pairs. Previous development pairs are excluded.
Sampled duplicate screening does not resolve original-source provenance.

Both local refinement arms use the same three query-relevant historical windows.
Each window is decoded at 4 FPS. Uniform refinement picks two fixed sub-second offsets.
Change-guided refinement picks the frames that bracket the largest low-resolution
RGB change. That score is a visual proxy and may respond to camera motion.
Both arms draw from the same 39 decoded history candidates, encode six selected
history frames for answering, and append four matched recent frames.
Ordinary 1 FPS uniform history is the practical reference.

The selector uses no gold answers or evidence intervals. Compare the robust paired
results against local uniform before advancing. The contrast matches window access
and sampled frames, while the change score incurs extra image processing.

After **TWINPAST_REFINEMENT_COMPLETE**, save the executed notebook and download
**HERMES_twinpast_refinement_evidence.zip**.

New candidate clips total 313 MiB.


In [1]:
!source /etc/network_turbo

设置成功!
注意：
1. 仅限学术用途和加速访问github/huggingface，不承诺稳定性
2. 开启加速后对访问其他资源如pip源等会*更慢*


In [2]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, time, zipfile, ast
from datetime import datetime, timezone

WORK = Path('/root/autodl-tmp/hermes_twinpast_refinement')
REUSE = Path('/root/autodl-tmp/hermes_causal_pilot')
BASE_REVISION = '2a1385e0abc94debb64761b0465713c799a81c1c'
MODEL_ID = 'llava-hf/llava-onevision-qwen2-7b-ov-hf'
MODEL_SHA = '0d50680527681998e456c7b78950205bedd8a068'
TF_SHA = '66bc4def9505fa7c7fe4aa7a248c34a026bb552b'
VALIDATE_ONLY = os.environ.get('HERMES_TWINPAST_REFINEMENT_VALIDATE_ONLY') == '1'
assert VALIDATE_ONLY or (WORK.is_absolute() and REUSE.is_absolute())
assert WORK != REUSE and not WORK.is_relative_to(REUSE) and not REUSE.is_relative_to(WORK)
REPO = WORK/'pinned_source'
VENV = WORK/'venv'
PYTHON = VENV/'bin/python'
RUNNER = WORK/'provenance/refinement_runner.py'
print('Plan: Up to 8 fresh pairs, 3 matched methods')
print('WORK:', WORK, '\nReuse candidate:', REUSE)
print('Mode:', 'CPU notebook validation (no network/GPU)' if VALIDATE_ONLY else 'AutoDL execution')
MEDIA_REUSE = Path('/root/autodl-tmp/twinpast-development')
PREVIOUS = WORK.with_name('hermes_twinpast_temporal_windows')


Plan: Up to 8 fresh pairs, 3 matched methods
WORK: /root/autodl-tmp/hermes_twinpast_refinement 
Reuse candidate: /root/autodl-tmp/hermes_causal_pilot
Mode: AutoDL execution


## 1. Verify runtime and materialize the experiment


In [3]:
def network_environment():
    env = os.environ.copy()
    if Path('/etc/network_turbo').is_file():
        proc = subprocess.run(['bash', '-lc', 'source /etc/network_turbo >/dev/null 2>&1; env -0'], capture_output=True)
        if proc.returncode == 0:
            for item in proc.stdout.split(b'\0'):
                if b'=' in item:
                    key, value = item.split(b'=', 1)
                    env[key.decode()] = value.decode()
    env.update(HF_HOME=str((REUSE if (REUSE/'cache/huggingface').is_dir() else WORK)/'cache/huggingface'), HF_HUB_DISABLE_XET='1',
               HF_HUB_ENABLE_HF_TRANSFER='0', TOKENIZERS_PARALLELISM='false',
               CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='8', GIT_LFS_SKIP_SMUDGE='1',
               UV_CACHE_DIR=str(WORK/'cache/uv'), UV_LINK_MODE='copy', TMPDIR=str(WORK),
               UV_DEFAULT_INDEX='https://pypi.tuna.tsinghua.edu.cn/simple',
               PYTORCH_ALLOC_CONF='expandable_segments:True')
    env.pop('PYTHONHOME', None)
    env['PYTHONPATH'] = ''
    return env

def logged(command, name, env=None, cwd=None):
    import queue, threading
    logfile = WORK/'logs'/name
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print('SETUP_STAGE', name, flush=True)
    with logfile.open('a', encoding='utf-8') as handle:
        proc = subprocess.Popen([str(x) for x in command], env=env, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        lines = queue.Queue()
        def collect():
            try:
                for line in proc.stdout:
                    lines.put(line)
            finally:
                lines.put(None)
        reader = threading.Thread(target=collect, daemon=True)
        reader.start()
        started = time.monotonic()
        while True:
            try:
                line = lines.get(timeout=30)
            except queue.Empty:
                line = f'COMMAND_ALIVE log={name} elapsed_min={(time.monotonic()-started)/60:.1f} (liveness only)\n'
            if line is None:
                break
            print(line, end='', flush=True)
            handle.write(line)
            handle.flush()
        reader.join(timeout=5)
        status = proc.wait()
        proc.stdout.close()
    if status:
        raise RuntimeError(f'Command failed ({status}); see {logfile}. Failure evidence, if created, stays in WORK.')

def probe(command, env=None, cwd=None):
    return subprocess.run([str(x) for x in command], env=env, cwd=cwd,
                          text=True, capture_output=True)

def package_environment(base, index='https://pypi.tuna.tsinghua.edu.cn/simple'):
    # AutoDL turbo is for GitHub/Hugging Face. Domestic package mirrors should
    # bypass those proxies, especially for large NVIDIA dependency wheels.
    direct = dict(base)
    for key in ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
                'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy'):
        direct.pop(key, None)
    direct.update(UV_HTTP_TIMEOUT='600', UV_HTTP_RETRIES='5',
                  UV_CONCURRENT_DOWNLOADS='2', UV_DEFAULT_INDEX=index,
                  PIP_DEFAULT_TIMEOUT='600', PIP_RETRIES='5')
    return direct

def install_packages(uv, python, packages, env, extra=()):
    errors = []
    for index in ('https://pypi.tuna.tsinghua.edu.cn/simple', 'https://mirrors.aliyun.com/pypi/simple'):
        try:
            logged(uv + ['pip', 'install', '--python', python, '--index-url', index]
                   + list(extra) + list(packages), 'setup.log', package_environment(env, index))
            return
        except RuntimeError as exc:
            errors.append(type(exc).__name__)
            print('Package download failed; trying alternate domestic index with cached files', flush=True)
    raise RuntimeError('Package installation failed on both indexes; cached files are preserved. See setup.log.')

if not VALIDATE_ONLY:
    assert sys.platform.startswith('linux'), 'Run this notebook in an AutoDL Linux container'
    WORK.mkdir(parents=True, exist_ok=True)
    ENV = network_environment()
    free = shutil.disk_usage(WORK).free / 1024**3
    minimum_free = 1 if (REUSE/'venv/bin/python').is_file() and (REUSE/'cache/huggingface').is_dir() else 18
    assert free >= minimum_free, f'Need {minimum_free} GiB free on the data disk; found {free:.1f} GiB'
    smi = probe(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], ENV)
    assert smi.returncode == 0 and smi.stdout.strip(), 'CUDA GPU is missing'
    print('GPU:', smi.stdout.strip(), '\nFree GiB:', round(free, 1))
    import re
    first_gpu = smi.stdout.splitlines()[0]
    assert float(re.search(r'(\d+(?:\.\d+)?)\s*MiB', first_gpu)[1]) >= 22528, 'Use a GPU with at least 22 GiB VRAM'
    # Never print the environment or proxy values.


GPU: NVIDIA GeForce RTX 4080 SUPER, 32760 MiB, 580.142 
Free GiB: 24.5


In [4]:
SOURCES = {'pinned_source/inference/llavaov_hermes.py': 'import re\nimport time\nimport torch\nimport torch.nn.functional as F\nimport types\nfrom transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor, DynamicCache\nfrom logzero import logger\n\nfrom inference.abstract_hermes import Abstract_Hermes\nfrom inference.reindex_1d import (\n    _get_rotary_module,\n    get_cache_seq_len,\n    contiguous_kv,\n    compute_cos_sin_for_positions,\n    rotary_delta,\n    apply_rotary_delta_to_keys_only,\n    apply_rotary_pos_emb_1d,\n)\n\n\nclass LlavaOneVision_Hermes(LlavaOnevisionForConditionalGeneration, Abstract_Hermes):\n    def __init__(self, config, processor, init_prompt_ids, kv_size, streaming=True):\n        super().__init__(processor, init_prompt_ids, kv_size)\n        self.streaming = streaming\n        \n        num_layers = config.num_hidden_layers if hasattr(config, \'num_hidden_layers\') else 32\n        self.num_layers = num_layers\n        \n        self.short_term_ratio = 0.3\n        self.long_term_ratio = 0.3\n        self.short_term_threshold = int(self.num_layers * self.short_term_ratio)\n        self.long_term_threshold = int(self.num_layers * (1 - self.long_term_ratio))\n        \n        self._position_ids_cache = [None for _ in range(num_layers)]\n        self.token_activity_cache = [None for _ in range(num_layers)]\n        \n        self._layer_position_ids = {}\n        self._hook_handles = []\n        self._register_forward_hooks()\n        \n        self.total_processed_frames = 0\n        \n        self._patch_rotary_embeddings()\n\n    def _patch_rotary_embeddings(self):\n        """Patch rotary embedding to support dynamic resizing based on logical position IDs"""\n        def patched_forward(rotary_self, x, seq_len=None):\n            required_len = getattr(rotary_self, \'_required_seq_len\', 0)\n            if seq_len is not None and required_len > seq_len:\n                seq_len = required_len\n            return rotary_self._original_forward(x, seq_len=seq_len)\n\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    rotary_emb = layer.self_attn.rotary_emb\n                    if not hasattr(rotary_emb, \'_original_forward\'):\n                        rotary_emb._original_forward = rotary_emb.forward\n                        rotary_emb.forward = types.MethodType(patched_forward, rotary_emb)\n                        rotary_emb._required_seq_len = 0\n\n    def _set_rotary_required_len(self, required_len):\n        """Set required sequence length for rotary embeddings"""\n        if hasattr(self.language_model.model, \'layers\'):\n            for layer in self.language_model.model.layers:\n                if hasattr(layer, \'self_attn\') and hasattr(layer.self_attn, \'rotary_emb\'):\n                    layer.self_attn.rotary_emb._required_seq_len = required_len\n\n    def _register_forward_hooks(self):\n        if not hasattr(self.language_model, \'model\') or not hasattr(self.language_model.model, \'layers\'):\n            return\n        \n        def create_hook(layer_idx):\n            def hook(module, args, kwargs):\n                if layer_idx in self._layer_position_ids:\n                    kwargs[\'position_ids\'] = self._layer_position_ids[layer_idx]\n                return args, kwargs\n            return hook\n        \n        for layer_idx, layer in enumerate(self.language_model.model.layers):\n            handle = layer.register_forward_pre_hook(create_hook(layer_idx), with_kwargs=True)\n            self._hook_handles.append(handle)\n    \n    def _clear_forward_hooks(self):\n        for handle in self._hook_handles:\n            handle.remove()\n        self._hook_handles = []\n\n    def _prune_activity_cache(self, keep_indices_per_layer):\n        if all(c is None for c in self.token_activity_cache):\n            return\n\n        for layer_idx in range(self.num_layers):\n            if self.token_activity_cache[layer_idx] is not None:\n                keep_indices = keep_indices_per_layer[layer_idx]\n                if not isinstance(keep_indices, torch.Tensor):\n                    keep_indices = torch.tensor(keep_indices, device=self.device, dtype=torch.long)\n                \n                curr_len = self.token_activity_cache[layer_idx].shape[0]\n                keep_indices = keep_indices[(keep_indices >= 0) & (keep_indices < curr_len)]\n                \n                if keep_indices.numel() > 0:\n                    self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][keep_indices]\n                else:\n                    self.token_activity_cache[layer_idx] = torch.zeros(0, device=self.device)\n\n    def _append_position_ids_layer(self, layer_idx: int, start: int, length: int):\n        device = self.device\n        new_pos = torch.arange(start, start + length, device=device, dtype=torch.long)\n        if self._position_ids_cache[layer_idx] is None:\n            self._position_ids_cache[layer_idx] = new_pos\n        else:\n            self._position_ids_cache[layer_idx] = torch.cat(\n                [self._position_ids_cache[layer_idx], new_pos], dim=0\n            )\n    \n    def _append_position_ids(self, start_per_layer, length: int):\n        if isinstance(start_per_layer, int):\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer, length)\n        else:\n            for layer_idx in range(self.num_layers):\n                self._append_position_ids_layer(layer_idx, start_per_layer[layer_idx], length)\n\n    def _get_cache_seq_len_per_layer(self) -> list:\n        if self.kv_cache is None:\n            return [0] * self.num_layers\n        \n        lengths = []\n        for layer_idx in range(len(self.kv_cache)):\n            k_layer, v_layer = self.kv_cache[layer_idx]\n            lengths.append(k_layer.shape[2])\n        return lengths\n\n    def _get_next_start_pos_per_layer(self) -> list:\n        next_pos = []\n        for layer_idx in range(self.num_layers):\n            if (layer_idx < len(self._position_ids_cache) and \n                self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > 0):\n                next_pos.append(self._position_ids_cache[layer_idx][-1].item() + 1)\n            else:\n                next_pos.append(0)\n        return next_pos\n    \n    def _sanitize_keep_indices(self, keep_indices_1d: torch.Tensor, seq_len: int) -> torch.Tensor:\n        keep_indices_1d = keep_indices_1d.to(self.device, dtype=torch.long)\n        keep_indices_1d = keep_indices_1d[(keep_indices_1d >= 0) & (keep_indices_1d < seq_len)]\n        if keep_indices_1d.numel() == 0:\n            return torch.tensor([0], device=self.device, dtype=torch.long)\n        keep_indices_1d = torch.unique(keep_indices_1d, sorted=True)\n        return keep_indices_1d\n\n    def _build_position_ids(self, past_len: int, q_len: int, batch: int) -> torch.Tensor:\n        pos_1d = torch.arange(past_len, past_len + q_len, device=self.device, dtype=torch.long)\n        position_ids = pos_1d.view(1, -1).expand(batch, -1)\n        return position_ids\n\n    @torch.inference_mode()\n    def _shrink_positions_and_rerotate_keys(self, keep_indices_per_layer):\n        device = self.device\n        \n        curr_lens = self._get_cache_seq_len_per_layer()\n        \n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is None or \n                self._position_ids_cache[layer_idx].numel() != curr_lens[layer_idx]):\n                self._position_ids_cache[layer_idx] = torch.arange(\n                    curr_lens[layer_idx], device=device, dtype=torch.long\n                )\n        max_pos_limit = getattr(self.language_model.config, "max_position_embeddings", 32768)\n        compact_threshold = max_pos_limit - 1024\n        \n        current_max_pos = 0\n        for cache in self._position_ids_cache:\n             if cache is not None and cache.numel() > 0:\n                 current_max_pos = max(current_max_pos, cache[-1].item())\n        \n        should_compact = (current_max_pos > compact_threshold) if self.streaming else True\n        \n        if should_compact:\n            logger.info(f"[Shrink] Max position {current_max_pos} > {compact_threshold}. Compacting position IDs (Shift Left).")\n        \n        old_position_ids_cache = [cache.clone() for cache in self._position_ids_cache]\n\n        sample_k = self.kv_cache[0][0]\n        dtype = sample_k.dtype\n        \n        new_kv_cache = []\n        \n        for layer_idx, (k_layer, v_layer) in enumerate(self.kv_cache):\n            \n            keep_indices_layer = keep_indices_per_layer[layer_idx]\n            if not isinstance(keep_indices_layer, torch.Tensor):\n                keep_indices_layer = torch.as_tensor(keep_indices_layer, dtype=torch.long, device=device)\n            \n            seq_len_layer = k_layer.shape[2]\n            safe_idx = self._sanitize_keep_indices(keep_indices_layer, seq_len_layer)\n            \n            if safe_idx.numel() == 0:\n                logger.warning(f"Layer {layer_idx}: After sanitization, keep_indices is empty; keeping first token")\n                safe_idx = torch.tensor([0], device=device, dtype=torch.long)\n\n            is_long_term = (layer_idx >= self.long_term_threshold)\n            \n            # === Extract Kept Tokens ===\n            k_kept = torch.index_select(k_layer, dim=2, index=safe_idx)\n            v_kept = torch.index_select(v_layer, dim=2, index=safe_idx)\n            old_pos_kept = old_position_ids_cache[layer_idx].index_select(0, safe_idx)\n            \n            # === Determine New Positions for Kept Tokens ===\n            if should_compact:\n                new_pos_kept = torch.arange(len(safe_idx), device=device, dtype=torch.long)\n            else:\n                new_pos_kept = old_pos_kept\n\n            # === Handle RoPE for Kept Tokens ===\n            if should_compact:\n                cos_old, sin_old = compute_cos_sin_for_positions(\n                    self.language_model, len(old_pos_kept), old_pos_kept, dtype, device\n                )\n                cos_new, sin_new = compute_cos_sin_for_positions(\n                    self.language_model, len(new_pos_kept), new_pos_kept, dtype, device\n                )\n                cos_delta, sin_delta = rotary_delta(\n                    cos_old, sin_old, cos_new, sin_new\n                )\n                k_kept_final = apply_rotary_delta_to_keys_only(k_kept, cos_delta, sin_delta)\n            else:\n                k_kept_final = k_kept\n            \n            # === Handle Folding (Summary Token) for Long-term Layers ===\n            if is_long_term:\n                mask = torch.ones(seq_len_layer, dtype=torch.bool, device=device)\n                mask[safe_idx] = False\n                prune_indices = torch.nonzero(mask).squeeze(1)\n                \n                if prune_indices.numel() > 0:\n                    k_pruned = torch.index_select(k_layer, dim=2, index=prune_indices)\n                    v_pruned = torch.index_select(v_layer, dim=2, index=prune_indices)\n                    \n                    v_summary = v_pruned.mean(dim=2, keepdim=True)\n                    \n                    old_pos_pruned = old_position_ids_cache[layer_idx].index_select(0, prune_indices)\n                    \n                    if should_compact:\n                        summary_pos_id = len(safe_idx)\n                    else:\n                        summary_pos_id = new_pos_kept[-1].item() + 1\n                    \n                    summary_pos_tensor = torch.tensor([summary_pos_id], device=device, dtype=torch.long)\n                    target_pos_pruned = summary_pos_tensor.repeat(old_pos_pruned.shape[0])\n                    \n                    cos_old, sin_old = compute_cos_sin_for_positions(\n                        self.language_model, len(old_pos_pruned), old_pos_pruned, dtype, device\n                    )\n                    cos_new, sin_new = compute_cos_sin_for_positions(\n                        self.language_model, len(target_pos_pruned), target_pos_pruned, dtype, device\n                    )\n                    \n                    cos_delta, sin_delta = rotary_delta(cos_old, sin_old, cos_new, sin_new)\n                    \n                    k_pruned_aligned = apply_rotary_delta_to_keys_only(k_pruned, cos_delta, sin_delta)\n                    \n                    k_summary_final = k_pruned_aligned.mean(dim=2, keepdim=True)\n                    \n                    k_final = torch.cat([k_kept_final, k_summary_final], dim=2)\n                    v_final = torch.cat([v_kept, v_summary], dim=2)\n                    \n                    new_pos_layer = torch.cat([new_pos_kept, summary_pos_tensor])\n                    \n                else:\n                    k_final = k_kept_final\n                    v_final = v_kept\n                    new_pos_layer = new_pos_kept\n            else:\n                k_final = k_kept_final\n                v_final = v_kept\n                new_pos_layer = new_pos_kept\n\n            new_kv_cache.append((k_final.contiguous(), v_final.contiguous()))\n            \n            self._position_ids_cache[layer_idx] = new_pos_layer.clone()\n        \n        self.kv_cache = new_kv_cache\n        \n        new_lens = self._get_cache_seq_len_per_layer()\n        logger.info(f"Layer-wise shrink completed. Lengths: min={min(new_lens)}, max={max(new_lens)}, "\n                   f"first={new_lens[0]}, last={new_lens[-1]}")\n    \n    @torch.inference_mode()\n    def predict_next_question(self):\n        """\n        Generates queries to drive attention calculation.\n        Includes Semantic Anchors for better coverage.\n        """\n        if hasattr(self, \'conv_history\') and len(self.conv_history) > 0:\n            last_q, last_a, last_options = self.conv_history[-1]\n            \n            option_match = re.match(r\'^\\s*(?:\\()?([A-Z])(?:\\))?\\.?\\s*$\', last_a)\n            if option_match and last_options is not None:\n                option_char = option_match.group(1)\n                option_idx = ord(option_char) - ord(\'A\')\n                if 0 <= option_idx < len(last_options):\n                    last_a = last_options[option_idx]\n\n            last_a = re.sub(r\'[A-Z]\\) \', \'\', last_a)\n\n            last_round_history = f"Question: {last_q} Answer: {last_a}"\n            \n            global_query = f"Context summary: {last_round_history}. Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            local_query = f"Find recent details related to: {last_round_history}. Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            \n        else:\n            global_query = (\n                "Summarize the video narrative, identifying main characters, key events, timeline changes, and the overall theme."\n            )\n            local_query = (\n                "Describe the current scene in detail, focusing on specific objects, fine-grained actions, and spatial relationships."\n            )\n            \n        print(f"Local question: {local_query}")\n        print(f"Global question: {global_query}")\n        return local_query, global_query\n    \n    @torch.inference_mode()\n    def encode_init_prompt(self):\n        super().encode_init_prompt()\n        self.total_processed_frames = 0\n        curr_lens = self._get_cache_seq_len_per_layer()\n        for layer_idx in range(self.num_layers):\n            self._position_ids_cache[layer_idx] = torch.arange(\n                curr_lens[layer_idx], device=self.device, dtype=torch.long\n            )\n    \n    def get_video_features(self, pixel_values_videos):\n            batch_size, frames, channels, height, width = pixel_values_videos.shape\n            pixel_values_videos = pixel_values_videos.view(batch_size * frames, channels, height, width)\n            video_features = self.vision_tower(pixel_values_videos, output_hidden_states=True)\n            selected_video_feature = video_features.hidden_states[self.config.vision_feature_layer]\n            if self.config.vision_feature_select_strategy == "default":\n                selected_video_feature = selected_video_feature[:, 1:]\n            elif self.config.vision_feature_select_strategy == "full":\n                selected_video_feature = selected_video_feature\n            video_features = self.multi_modal_projector(selected_video_feature)\n            video_features = self.apply_pooling(video_features)\n            frames_after_merge = video_features.shape[0]\n            video_features = video_features.reshape(batch_size, frames_after_merge * video_features.shape[1], -1)  # (B, Nv*196, D)\n            return video_features\n    \n    @torch.inference_mode()\n    def encode_video_chunk(self, video_chunk):\n        if video_chunk is None or (hasattr(video_chunk, "shape") and video_chunk.shape[0] == 0):\n            return\n\n        if len(video_chunk.shape) == 4 and video_chunk.shape[-1] == 3:\n            video_chunk = video_chunk.permute(0, 3, 1, 2)\n\n        pixel_values_videos = self.processor.video_processor(\n            video_chunk, return_tensors="pt"\n        ).pixel_values_videos.to(self.device, self.dtype)\n        video_features = self.get_video_features(pixel_values_videos)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = video_features.shape[1]\n        batch = video_features.shape[0]\n        \n        max_start_pos = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start_pos + q_len)\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        default_position_ids = self._build_position_ids(start_pos_per_layer[0], q_len, batch)\n        \n        out = self.language_model(\n            inputs_embeds=video_features,\n            past_key_values=self.kv_cache,\n            use_cache=True,\n            return_dict=True,\n            position_ids=default_position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        self.kv_cache = out.past_key_values\n        self.kv_cache = contiguous_kv(self.kv_cache)\n\n        self._append_position_ids(start_pos_per_layer, q_len)\n        self.last_encoded_frames = video_chunk.shape[0]\n        self.total_processed_frames += video_chunk.shape[0]\n        \n        self._layer_position_ids.clear()\n\n    @torch.inference_mode()\n    def apply_kv_cache_pruning_strict(self, keep_indices_all_layers):\n        if self.kv_cache is None:\n            logger.warning("No KV-Cache to prune")\n            return\n        if not keep_indices_all_layers or len(keep_indices_all_layers[0]) == 0:\n            logger.warning("Empty keep_indices; skip pruning")\n            return\n\n        self._shrink_positions_and_rerotate_keys(keep_indices_all_layers)\n        self._prune_activity_cache(keep_indices_all_layers)\n        \n        logger.info(f"Strict-shrunk KV cache. New length: {get_cache_seq_len(self.kv_cache)}")\n\n    def allocate_budget_by_depth(self, total_budget, num_layers):\n        """Allocate budget uniformly across layers."""\n        budget_per_layer = [total_budget // num_layers] * num_layers\n        diff = total_budget - sum(budget_per_layer)\n        budget_per_layer[-1] += diff\n        return budget_per_layer\n\n    def _compute_attention_scores_manually(self, input_ids, past_key_values):\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len = input_ids.shape[1]\n        batch = input_ids.shape[0]\n        \n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        \n        config = self.language_model.config\n        num_layers = config.num_hidden_layers\n        num_heads = config.num_attention_heads\n        num_key_value_heads = config.num_key_value_heads\n        head_dim = config.hidden_size // num_heads\n        \n        attention_weights_list = []\n        \n        hidden_states = inputs_embeds\n        \n        for layer_idx in range(num_layers):\n            layer = self.language_model.model.layers[layer_idx]\n            \n            past_k, past_v = past_key_values[layer_idx]\n            \n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len, batch)\n            \n            hidden_states_norm = layer.input_layernorm(hidden_states)\n            \n            attn = layer.self_attn\n            \n            query_states = attn.q_proj(hidden_states_norm)\n            query_states = query_states.view(batch, q_len, num_heads, head_dim).transpose(1, 2)\n            \n            key_states = attn.k_proj(hidden_states_norm)\n            key_states = key_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            value_states = attn.v_proj(hidden_states_norm)\n            value_states = value_states.view(batch, q_len, num_key_value_heads, head_dim).transpose(1, 2)\n            \n            kv_seq_len = position_ids.max().item() + 1\n            rotary_emb = _get_rotary_module(self.language_model)\n            cos, sin = rotary_emb(value_states, kv_seq_len)\n            cos = cos[position_ids].unsqueeze(1)\n            sin = sin[position_ids].unsqueeze(1)\n            \n            query_states = apply_rotary_pos_emb_1d(query_states, cos, sin)\n            key_states = apply_rotary_pos_emb_1d(key_states, cos, sin)\n            \n            key_states = torch.cat([past_k, key_states], dim=2)\n            value_states = torch.cat([past_v, value_states], dim=2)\n            \n            if num_key_value_heads != num_heads:\n                n_rep = num_heads // num_key_value_heads\n                key_states = torch.repeat_interleave(key_states, n_rep, dim=1)\n                value_states = torch.repeat_interleave(value_states, n_rep, dim=1)\n            \n            attn_weights = torch.matmul(query_states.float(), key_states.float().transpose(-2, -1)) / (head_dim ** 0.5)\n            attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n            attention_weights_list.append(attn_weights)\n        \n        return attention_weights_list\n\n    def prune_kv_cache_by_attention(self, attn_weights_local, attn_weights_global, attn_weights_mixed, num_keep=3000):\n        device = self.device\n        visual_start_idx = self.visual_start_idx\n        num_layers = len(attn_weights_local)\n        \n        curr_cache_lens = self._get_cache_seq_len_per_layer()\n        \n        question_len_local = attn_weights_local[0].shape[2]\n        question_len_global = attn_weights_global[0].shape[2]\n        question_len_mixed = attn_weights_mixed[0].shape[2]\n\n        total_budget = num_keep * num_layers\n        budget_per_layer = self.allocate_budget_by_depth(total_budget, num_layers)\n        \n        keep_indices_all_layers = []\n\n        layer_raw_scores = []\n        layer_configs = []\n\n        for layer_idx in range(len(attn_weights_local)):\n            curr_len = curr_cache_lens[layer_idx]\n            if self.token_activity_cache[layer_idx] is None:\n                self.token_activity_cache[layer_idx] = torch.zeros(curr_len, device=device, dtype=torch.float32)\n            elif self.token_activity_cache[layer_idx].shape[0] < curr_len:\n                diff = curr_len - self.token_activity_cache[layer_idx].shape[0]\n                zeros = torch.zeros(diff, device=device, dtype=torch.float32)\n                self.token_activity_cache[layer_idx] = torch.cat([self.token_activity_cache[layer_idx], zeros])\n            \n            if self.token_activity_cache[layer_idx].shape[0] > curr_len:\n                 self.token_activity_cache[layer_idx] = self.token_activity_cache[layer_idx][:curr_len]\n\n            # Memory hierarchy mode\n            if layer_idx < self.short_term_threshold:\n                layer_type = "short-term"\n                layer_attn_weights = attn_weights_local[layer_idx]\n                question_len = question_len_local\n                layer_recency_alpha = 1\n                k = 20\n                \n            elif layer_idx >= self.long_term_threshold:\n                layer_type = "long-term"\n                layer_attn_weights = attn_weights_global[layer_idx]\n                question_len = question_len_global\n                layer_recency_alpha = 0\n                k = 0.0\n                \n            else:\n                layer_type = "mid-term"\n                layer_attn_weights = attn_weights_mixed[layer_idx]\n                question_len = question_len_mixed\n                progress = (layer_idx - self.short_term_threshold) / (self.long_term_threshold - self.short_term_threshold)\n                layer_recency_alpha = 1.0 - 0.9 * progress\n                k = 20 - 19.5 * progress\n            \n            visual_attn_weights = layer_attn_weights[0].mean(dim=0)[:,visual_start_idx:-1*question_len].mean(dim=0)\n            num_visual_tokens = visual_attn_weights.shape[0]\n            \n            end_idx = visual_start_idx + num_visual_tokens\n            if end_idx <= self.token_activity_cache[layer_idx].shape[0]:\n                self.token_activity_cache[layer_idx][visual_start_idx:end_idx] += visual_attn_weights\n            \n            layer_budget = budget_per_layer[layer_idx]\n            \n            if layer_type == \'long-term\':\n                layer_budget = max(0, layer_budget - 1)\n\n            positions = torch.arange(num_visual_tokens, device=device, dtype=torch.float32)\n            time_distances = (num_visual_tokens - 1 - positions) / max(num_visual_tokens - 1, 1)\n            \n            # Exponential decay\n            recency_weights = torch.exp(-k * time_distances)\n            \n            attn_norm = (visual_attn_weights - visual_attn_weights.min()) / \\\n                        (visual_attn_weights.max() - visual_attn_weights.min() + 1e-6)\n            recency_norm = (recency_weights - recency_weights.min()) / \\\n                        (recency_weights.max() - recency_weights.min() + 1e-6)\n            \n            raw_score = attn_norm * (1 - layer_recency_alpha) + recency_norm * layer_recency_alpha\n            \n            layer_raw_scores.append(raw_score)\n            layer_configs.append({\n                \'budget\': min(layer_budget, num_visual_tokens),\n                \'layer_type\': layer_type,\n                \'visual_start_idx\': visual_start_idx\n            })\n        \n        refined_scores = [s.clone() for s in layer_raw_scores]\n        \n        for i in range(len(refined_scores) - 2, -1, -1):\n            current_type = layer_configs[i][\'layer_type\']\n            \n            # TODO: change hard code here\n            if current_type == \'long-term\':\n                gamma = 0.4  \n            elif current_type == \'mid-term\':\n                gamma = 0.3  \n            else:\n                gamma = 0.1\n            \n            score_current = refined_scores[i]\n            score_next = refined_scores[i+1]\n            \n            if score_current.shape[0] != score_next.shape[0]:\n                score_next_reshaped = score_next.view(1, 1, -1)\n                score_next_interp = F.interpolate(\n                    score_next_reshaped, \n                    size=score_current.shape[0], \n                    mode=\'linear\', \n                    align_corners=False\n                ).view(-1)\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next_interp\n            else:\n                refined_scores[i] = (1 - gamma) * score_current + gamma * score_next\n\n        for layer_idx, score in enumerate(refined_scores):\n            config = layer_configs[layer_idx]\n            actual_num_keep = config[\'budget\']\n            start_idx = config[\'visual_start_idx\']\n            \n            topk_indices_relative = torch.topk(score, actual_num_keep, sorted=False)[1]\n            topk_indices_absolute = topk_indices_relative + start_idx\n            topk_indices_absolute_sorted = torch.sort(topk_indices_absolute)[0]\n            \n            keep_indices = torch.cat([\n                torch.arange(start_idx, device=device), \n                topk_indices_absolute_sorted\n            ]).tolist()\n            \n            keep_indices_all_layers.append(keep_indices)\n        \n        return keep_indices_all_layers\n    \n    @torch.inference_mode()\n    def pseudo_forward(self, local_question=None, global_question=None):\n        device = self.device\n        \n        if local_question is None:\n            local_question = "What is happening in the video?"\n        if global_question is None:\n            global_question = "What is the main topic of the video?"\n        \n        local_input_ids = self.processor.tokenizer(local_question).input_ids\n        local_input_ids = torch.as_tensor([local_input_ids], device=device, dtype=torch.int)\n\n        start_pos_per_layer = self._get_next_start_pos_per_layer()\n        q_len_local = local_input_ids.shape[1]\n        batch = local_input_ids.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_local, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_local = self._build_position_ids(start_pos_per_layer[0], q_len_local, batch)\n\n        max_start = max(start_pos_per_layer) if start_pos_per_layer else 0\n        self._set_rotary_required_len(max_start + q_len_local)\n\n        use_flash_attn = (hasattr(self.language_model.config, \'_attn_implementation\') and \n                        self.language_model.config._attn_implementation in ["flash_attention_2", "sdpa"])\n\n        if use_flash_attn:\n            attn_weights_local = self._compute_attention_scores_manually(local_input_ids, self.kv_cache)\n        else:\n            out_local = self.language_model(\n                input_ids=local_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_local,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_local = out_local.attentions\n        \n        global_input_ids = self.processor.tokenizer(global_question).input_ids\n        global_input_ids = torch.as_tensor([global_input_ids], device=device, dtype=torch.int)\n        \n        q_len_global = global_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_global, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_global = self._build_position_ids(start_pos_per_layer[0], q_len_global, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_global)\n        \n        if use_flash_attn:\n            attn_weights_global = self._compute_attention_scores_manually(global_input_ids, self.kv_cache)\n        else:\n            out_global = self.language_model(\n                input_ids=global_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_global,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_global = out_global.attentions\n        \n        mixed_question = local_question + "; " + global_question\n        mixed_input_ids = self.processor.tokenizer(mixed_question).input_ids\n        mixed_input_ids = torch.as_tensor([mixed_input_ids], device=device, dtype=torch.int)\n        \n        q_len_mixed = mixed_input_ids.shape[1]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_per_layer[layer_idx], q_len_mixed, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids_mixed = self._build_position_ids(start_pos_per_layer[0], q_len_mixed, batch)\n        \n        self._set_rotary_required_len(max_start + q_len_mixed)\n        \n        if use_flash_attn:\n            attn_weights_mixed = self._compute_attention_scores_manually(mixed_input_ids, self.kv_cache)\n        else:\n            out_mixed = self.language_model(\n                input_ids=mixed_input_ids,\n                use_cache=False,\n                past_key_values=self.kv_cache,\n                output_attentions=True,\n                position_ids=position_ids_mixed,\n                cache_position=None,\n                attention_mask=None,\n            )\n            attn_weights_mixed = out_mixed.attentions\n        \n        self._layer_position_ids.clear()\n        \n        print(f"GPU memory usage: {self.get_gpu_memory_usage_gb()} GB")\n        current_k_states_len = self.kv_cache[0][0].shape[2]\n        \n        keep_indices_all_layers = self.prune_kv_cache_by_attention(\n            attn_weights_local, attn_weights_global, attn_weights_mixed, \n            num_keep=self.kv_size\n        )\n        \n        if current_k_states_len > self.kv_size:\n            print(f"Applying KV-Cache compression due to k_states > {self.kv_size}")\n            self.apply_kv_cache_pruning_strict(keep_indices_all_layers)\n    \n    @torch.inference_mode()\n    def predict_and_compress(self):\n        local_question, global_question = self.predict_next_question()\n        self.pseudo_forward(local_question, global_question)\n\n    @torch.inference_mode()\n    def question_answering(self, input_text, max_new_tokens=128, temperature=0, repetition_penalty=1.1, pseudo_forward=False):\n        device = self.device\n        stop_token_ids = [self.processor.tokenizer.eos_token_id]\n        output_ids = []\n\n        start_time = time.perf_counter()\n        \n        prompt = input_text[\'prompt\']\n        input_ids = self.processor.tokenizer(prompt).input_ids\n        input_ids = torch.as_tensor([input_ids], device=device)\n\n        past_lens_prefill = self._get_cache_seq_len_per_layer()\n        start_pos_prefill = self._get_next_start_pos_per_layer()\n\n        inputs_embeds = self.get_input_embeddings()(input_ids)\n        q_len_prefill = inputs_embeds.shape[1]\n        batch = inputs_embeds.shape[0]\n        \n        self._layer_position_ids.clear()\n        for layer_idx in range(self.num_layers):\n            position_ids = self._build_position_ids(start_pos_prefill[layer_idx], q_len_prefill, batch)\n            self._layer_position_ids[layer_idx] = position_ids\n        \n        position_ids = self._build_position_ids(start_pos_prefill[0], q_len_prefill, batch)\n\n        max_start_pos = max(start_pos_prefill) if start_pos_prefill else 0\n        self._set_rotary_required_len(max_start_pos + q_len_prefill)\n\n        out = self.language_model(\n            inputs_embeds=inputs_embeds,\n            use_cache=True,\n            past_key_values=self.kv_cache,\n            position_ids=position_ids,\n            cache_position=None,\n            attention_mask=None,\n        )\n        past_key_values = out.past_key_values\n        logits = out.logits\n        \n        self._append_position_ids(start_pos_prefill, q_len_prefill)\n        self._layer_position_ids.clear()\n\n        for step in range(max_new_tokens):\n            last_token_logits = logits[0, -1, :]\n\n            if repetition_penalty != 1.0 and len(output_ids) > 0:\n                for token_id in set(output_ids):\n                    if last_token_logits[token_id] < 0:\n                        last_token_logits[token_id] *= repetition_penalty\n                    else:\n                        last_token_logits[token_id] /= repetition_penalty\n\n            if temperature == 0.0:\n                _, indices = torch.topk(last_token_logits, 1)\n                token = int(indices[0])\n            else:\n                scaled_logits = last_token_logits / temperature\n                scaled_logits = torch.nan_to_num(\n                    scaled_logits, nan=-float(\'inf\'), posinf=float(\'inf\'), neginf=-float(\'inf\')\n                )\n                probs = F.softmax(scaled_logits, dim=-1)\n                probs = torch.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)\n                probs_sum = probs.sum()\n                if probs_sum > 0:\n                    probs = probs / probs_sum\n                    token = torch.multinomial(probs, num_samples=1).item()\n                else:\n                    _, indices = torch.topk(last_token_logits, 1)\n                    token = int(indices[0])\n\n            output_ids.append(token)\n            if token in stop_token_ids:\n                break\n\n            curr_start_pos = self._get_next_start_pos_per_layer()\n            \n            max_curr_start = max(curr_start_pos) if curr_start_pos else 0\n            self._set_rotary_required_len(max_curr_start + 1)\n            \n            self._layer_position_ids.clear()\n            for layer_idx in range(self.num_layers):\n                pos_step = torch.tensor([curr_start_pos[layer_idx]], device=device, dtype=torch.long)\n                position_ids_layer = pos_step.view(1, 1)\n                self._layer_position_ids[layer_idx] = position_ids_layer\n            \n            pos_step = torch.tensor([curr_start_pos[0]], device=device, dtype=torch.long)\n            position_ids = pos_step.view(1, 1)\n\n            out = self.language_model(\n                input_ids=torch.as_tensor([[token]], device=device),\n                use_cache=True,\n                past_key_values=past_key_values,\n                position_ids=position_ids,\n                cache_position=None,\n                attention_mask=None,\n            )\n            \n            if (not pseudo_forward) and (step == 0):\n                end_time = time.perf_counter()\n                print(f"TTFT: {end_time - start_time} seconds")\n                \n            logits = out.logits\n            past_key_values = out.past_key_values\n\n            self._append_position_ids(curr_start_pos, 1)\n            self._layer_position_ids.clear()\n\n        output = self.processor.tokenizer.decode(\n            output_ids,\n            skip_special_tokens=True,\n            spaces_between_special_tokens=False,\n            clean_up_tokenization_spaces=True,\n        )\n\n        if not pseudo_forward:\n            current_question = input_text[\'question\']\n            current_options = None\n            formatted_question = input_text.get(\'formatted_question\', None)\n            if formatted_question:\n                option_matches = re.findall(r\'\\([A-Z]\\)\\s*(.+?)(?=\\n\\([A-Z]\\)|\\nThe best answer|\\n*$)\', formatted_question, re.DOTALL)\n                if option_matches:\n                    current_options = [opt.strip() for opt in option_matches]\n            self.conv_history.append((current_question, output, current_options))\n            logger.info(f"Saved conversation to history. Total conversations: {len(self.conv_history)}")\n\n        self._truncate_kv_cache(past_lens_prefill)\n        for layer_idx in range(self.num_layers):\n            if (self._position_ids_cache[layer_idx] is not None and \n                self._position_ids_cache[layer_idx].numel() > past_lens_prefill[layer_idx]):\n                self._position_ids_cache[layer_idx] = self._position_ids_cache[layer_idx][\n                    :past_lens_prefill[layer_idx]\n                ].contiguous()\n\n        new_lens = self._get_cache_seq_len_per_layer()\n        print(f"Answering Cache lengths: min={min(new_lens)}, max={max(new_lens)}")\n        return output\n    \n    def _truncate_kv_cache(self, target_lengths):\n        if self.kv_cache is None:\n            return\n        \n        truncated_cache = []\n        for layer_idx, (k_cache, v_cache) in enumerate(self.kv_cache):\n            if isinstance(target_lengths, int):\n                target_len = target_lengths\n            else:\n                target_len = target_lengths[layer_idx]\n            \n            truncated_k = k_cache[:, :, :target_len, :]\n            truncated_v = v_cache[:, :, :target_len, :]\n            truncated_cache.append((truncated_k, truncated_v))\n        \n        if isinstance(self.kv_cache, DynamicCache):\n            self.kv_cache = DynamicCache.from_legacy_cache(truncated_cache)\n        else:\n            self.kv_cache = truncated_cache\n\n\ndef load_model(model_path=\'llava-onevision-qwen2-7b-ov-hf\',\n               n_init=None, kv_size=None, streaming=True, device="cuda", sample_fps=0.5):\n    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)\n    processor.tokenizer.padding_side = \'left\'\n    \n    system_prompt = \'<|im_start|>system \\nYou are a helpful assistant.<|im_end|><|im_start|>user \'\n    init_prompt_ids = processor.tokenizer(system_prompt, return_tensors="pt").input_ids.to(device)\n    \n    base_model = LlavaOnevisionForConditionalGeneration.from_pretrained(\n        model_path,\n        device_map="auto",\n        torch_dtype=torch.float16,\n        trust_remote_code=True,\n    )\n\n    model = LlavaOneVision_Hermes.__new__(LlavaOneVision_Hermes)\n    model.__dict__ = base_model.__dict__.copy()\n\n    Abstract_Hermes.__init__(\n        model, \n        processor, \n        init_prompt_ids.tolist(), \n        kv_size,\n    )\n    model.streaming = streaming\n    \n    num_layers = base_model.language_model.config.num_hidden_layers\n    model.num_layers = num_layers\n    model._position_ids_cache = [None for _ in range(num_layers)]\n    \n    model.short_term_ratio = 0.1\n    model.long_term_ratio = 0.3\n    model.short_term_threshold = int(model.num_layers * model.short_term_ratio)\n    model.long_term_threshold = int(model.num_layers * (1 - model.long_term_ratio))\n    model.token_activity_cache = [None for _ in range(num_layers)]\n    \n    model.total_processed_frames = 0\n    \n    model._layer_position_ids = {}\n    model._hook_handles = []\n    model._register_forward_hooks()\n    model._patch_rotary_embeddings()\n    \n    logger.info(f\'n_init: {init_prompt_ids.shape[1] if n_init is None else n_init}\')\n    logger.info(f\'kv_size: {kv_size}\')\n\n    model.eval()\n\n    return model, processor\n', 'pinned_source/inference/abstract_hermes.py': 'import torch\n\nclass Abstract_Hermes:\n    kv_cache = None\n\n    def __init__(self, processor, init_prompt_ids, kv_size):\n        self.processor = processor\n        self.init_prompt_ids = init_prompt_ids\n        self.kv_size = kv_size\n        self.last_encoded_frames = 0\n        self.visual_start_idx = 14\n        self.conv_history = []\n        \n    def clear_cache(self):\n        self.kv_cache = None\n        self.conv_history = []\n        torch.cuda.empty_cache()\n        torch.cuda.ipc_collect()\n\n    def encode_init_prompt(self):\n        if not isinstance(self.init_prompt_ids, torch.Tensor):\n            self.init_prompt_ids = torch.as_tensor(self.init_prompt_ids, device=self.device)\n        output = self.language_model(input_ids=self.init_prompt_ids, use_cache=True, return_dict=True)\n        self.kv_cache = output.past_key_values\n        self.visual_start_idx = self.kv_cache[0][0].shape[2]\n\n    def get_prompt(self, query, mc=False):\n        prompt = f"\\n{query}<|im_end|><|im_start|>assistant\\n"\n        \n        if mc:\n            prompt += \'Best option: (\'\n        return prompt\n\n    def get_gpu_memory_usage_gb(self):\n        return torch.cuda.max_memory_allocated() / (1024**3)', 'pinned_source/inference/reindex_1d.py': 'import torch\nfrom typing import Tuple\nfrom transformers import DynamicCache\n\n\ndef get_cache_seq_len(past_key_values) -> int:\n    if past_key_values is None:\n        return 0\n    if isinstance(past_key_values, DynamicCache):\n        return past_key_values.get_seq_length()\n    return past_key_values[0][0].shape[2]\n\n\ndef contiguous_kv(past_key_values):\n    new_legacy = []\n    for (k_layer, v_layer) in past_key_values:\n        new_legacy.append((k_layer.contiguous(), v_layer.contiguous()))\n    return new_legacy\n\n\ndef _get_rotary_module(llm) -> torch.nn.Module:\n    if hasattr(llm, "rotary_emb"):\n        return llm.rotary_emb\n    if hasattr(llm, "model") and hasattr(llm.model, "rotary_emb"):\n        return llm.model.rotary_emb\n\n    if hasattr(llm, "model") and hasattr(llm.model, "layers"):\n        if len(llm.model.layers) > 0 and hasattr(llm.model.layers[0], "self_attn"):\n            if hasattr(llm.model.layers[0].self_attn, "rotary_emb"):\n                return llm.model.layers[0].self_attn.rotary_emb\n    raise AttributeError("Cannot find rotary_emb module on language_model")\n\n\ndef rotate_half(x):\n    x1 = x[..., : x.shape[-1] // 2]\n    x2 = x[..., x.shape[-1] // 2 :]\n    return torch.cat((-x2, x1), dim=-1)\n\n\ndef apply_rotary_pos_emb_1d(x, cos, sin):\n    if cos.dim() == 3 and x.dim() == 4:\n        cos = cos.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n        sin = sin.unsqueeze(1)  # [batch, 1, seq_len, head_dim]\n    \n    x_embed = (x * cos) + (rotate_half(x) * sin)\n    return x_embed\n\n\ndef rotary_delta(cos_old, sin_old, cos_new, sin_new):\n    cos_delta = cos_new * cos_old + sin_new * sin_old\n    sin_delta = sin_new * cos_old - cos_new * sin_old\n    return cos_delta, sin_delta\n\n\n@torch.inference_mode()\ndef compute_cos_sin_for_positions(\n    llm, \n    seq_len: int, \n    position_ids_1d: torch.Tensor, \n    dtype: torch.dtype, \n    device: torch.device\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    rotary_emb = _get_rotary_module(llm)\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    \n    position_ids = position_ids_1d.view(1, -1).to(device)\n    \n    dummy_h = torch.zeros((1, seq_len, hidden_size), device=device, dtype=dtype)\n    \n    \n    seq_len_rope = position_ids.max().item() + 1\n    cos, sin = rotary_emb(dummy_h, seq_len=seq_len_rope)\n    cos = cos[position_ids]\n    sin = sin[position_ids]\n    \n    cos = cos.to(dtype)\n    sin = sin.to(dtype)\n    \n    return cos, sin\n\n@torch.inference_mode()\ndef compute_cos_sin_with_custom_base(\n    llm,\n    seq_len: int,\n    position_ids_1d: torch.Tensor,\n    dtype: torch.dtype,\n    device: torch.device,\n    rope_base: float = 10000.0\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    \n    config = llm.config if hasattr(llm, "config") else llm.model.config\n    hidden_size = getattr(config, "hidden_size", 4096)\n    num_attention_heads = getattr(config, "num_attention_heads", 32)\n    head_dim = hidden_size // num_attention_heads\n    \n    \n    inv_freq = 1.0 / (rope_base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))\n    \n    freqs = torch.outer(position_ids_1d.float(), inv_freq)\n    \n    \n    emb = torch.cat([freqs, freqs], dim=-1)\n    \n    cos = emb.cos().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    sin = emb.sin().unsqueeze(0).to(dtype)  # [1, seq_len, head_dim]\n    \n    return cos, sin\n\n\ndef get_layer_rope_base(\n    layer_idx: int,\n    num_layers: int,\n    short_term_ratio: float = 0.3,\n    long_term_ratio: float = 0.3,\n    short_term_base: float = 10000.0,\n    mid_term_base_range: Tuple[float, float] = (10000.0, 50000.0),\n    long_term_base: float = 50000.0,\n) -> float:\n    \n    short_term_threshold = int(num_layers * short_term_ratio)\n    long_term_threshold = int(num_layers * (1 - long_term_ratio))\n    \n    if layer_idx < short_term_threshold:\n        \n        return short_term_base\n    elif layer_idx >= long_term_threshold:\n        \n        return long_term_base\n    else:\n        \n        mid_start = short_term_threshold\n        mid_end = long_term_threshold\n        progress = (layer_idx - mid_start) / max(mid_end - mid_start, 1)\n        base_start, base_end = mid_term_base_range\n        return base_start + (base_end - base_start) * progress\n\n\n@torch.inference_mode()\ndef apply_rotary_delta_to_keys_only(\n    key_states: torch.Tensor, \n    cos_delta: torch.Tensor, \n    sin_delta: torch.Tensor\n) -> torch.Tensor:\n    \n    k_rot = apply_rotary_pos_emb_1d(key_states, cos_delta, sin_delta)\n    return k_rot\n\n', 'provenance/twinpast_base.py': '"""Run the frozen, annotation-guided TwinPast development diagnostic."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport statistics\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\nARTIFACT = "HERMES_twinpast_diagnostic_evidence.zip"\nARMS = ("question_only", "recent_four", "history_a", "history_b")\nORDERS = ("canonical", "reversed")\nLETTERS = "ABCDEFGH"\nHOSTS = ("huggingface.co", "hf-mirror.com")\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef digest(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef atomic(path, data):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef fetch(bundle, relative, dest, expected_sha, size=None, reuse=None):\n    import requests\n\n    if dest.is_file() and digest(dest) == expected_sha:\n        return\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temporary = dest.with_suffix(".part")\n    if reuse and reuse.is_file() and digest(reuse) == expected_sha:\n        shutil.copyfile(reuse, temporary)\n        temporary.replace(dest)\n        print(f"MEDIA_REUSED {relative}", flush=True)\n        return\n    cap = size if size is not None else 2 * 1024**2\n    remaining_attempts = [(host, direct) for direct in (False, True) for host in HOSTS]\n    for host, direct in remaining_attempts:\n        url = f"https://{host}/datasets/{bundle[\'manifest\'][\'dataset\']}/resolve/{bundle[\'manifest\'][\'revision\']}/{relative}"\n        try:\n            with requests.Session() as session:\n                session.trust_env = not direct\n                with session.get(url, stream=True, timeout=(8, 15)) as response, temporary.open("wb") as stream:\n                    response.raise_for_status()\n                    start, last_report, count = time.monotonic(), time.monotonic(), 0\n                    for block in response.iter_content(128 * 1024):\n                        count += len(block)\n                        now = time.monotonic()\n                        if count > cap or now - start > 120:\n                            raise ValueError("Download exceeded size or elapsed-time bound")\n                        stream.write(block)\n                        if now - last_report >= 8:\n                            print(f"MEDIA_PROGRESS {relative} {count / 1024**2:.1f} MiB", flush=True)\n                            last_report = now\n            if (size is not None and count != size) or digest(temporary) != expected_sha:\n                raise ValueError("Downloaded size or SHA-256 does not match")\n            temporary.replace(dest)\n            print(f"MEDIA_READY {relative} {count / 1024**2:.1f} MiB", flush=True)\n            return\n        except (requests.RequestException, OSError, ValueError) as exc:\n            print(f"MEDIA_RETRY {relative} host={host} direct={direct} reason={type(exc).__name__}", flush=True)\n    raise RuntimeError(f"Could not acquire {relative}. Verified completed files remain available for rerun.")\n\n\ndef cases(bundle):\n    plans = {row["pair_id"]: row for row in bundle["manifest"]["selected_pairs"]}\n    visual = {row["pair_id"]: row for row in bundle["visual"]}\n    media = {row["pair_id"]: row for row in bundle["media"]}\n    assert len(plans) == len(visual) == len(media) == 12 and plans.keys() == visual.keys() == media.keys()\n    selected = []\n    for pair_id, plan in plans.items():\n        if visual[pair_id]["status"] == "clear_contrast":\n            checked = media[pair_id]\n            assert checked["status"] == "checked" and checked["task"] == plan["task"]\n            assert checked["grid_s"] == sorted(set(checked["grid_s"])) and len(checked["grid_s"]) == 6\n            assert checked["grid_s"][-4:] == plan["latest_four_grid_s"]\n            assert checked["grid_s"][-1] < plan["query_time_s"]\n            for branch in "ab":\n                assert all(checked["decoded"][branch]["early_samples_inside"])\n            assert checked["decoded"]["a"]["decoded_rgb_sha256"][-4:] == checked["decoded"]["b"]["decoded_rgb_sha256"][-4:]\n            selected.append((plan, checked, visual[pair_id]))\n    assert len(selected) == 9\n    return selected\n\n\ndef annotation_rows(bundle, raw):\n    assert sha(raw) == bundle["manifest"]["annotation_sha256"], "Pinned annotations changed"\n    rows = [json.loads(line) for line in raw.splitlines() if line.strip()]\n    result = {row["pair_id"]: row for row in rows}\n    assert len(rows) == len(result) == bundle["manifest"]["dataset_pairs"]\n    for plan, media, visual in cases(bundle):\n        row = result[plan["pair_id"]]\n        assert all(row[field] == plan[field] for field in ("video_a", "video_b", "task"))\n        assert row["t_query"] == plan["query_time_s"] and row["question"].strip()\n        choices = row["options"]\n        assert 2 <= len(choices) <= len(LETTERS) and all(c.strip() for c in choices)\n        assert len({c.strip().casefold() for c in choices}) == len(choices)\n        assert row["answer_a"] != row["answer_b"]\n        for branch in "ab":\n            assert row[f"answer_{branch}"] in LETTERS[:len(choices)]\n            assert row[f"answer_{branch}"] == visual[f"observed_option_{branch}"]\n            seconds = media["decoded"][branch]["frame_start_s"]\n            assert all(t < row["t_query"] for t in seconds)\n            assert all(any(lo - 1e-6 <= t <= hi + 1e-6\n                           for lo, hi in row["evidence_intervals"][branch.upper()]) for t in seconds[:2])\n    return result\n\n\ndef decode(path, recorded):\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    frames = reader.get_batch(recorded["source_frames"]).asnumpy()\n    assert [sha(frame.tobytes()) for frame in frames] == recorded["decoded_rgb_sha256"], (\n        f"Decoded frames differ from the frozen audit for {path.name}")\n    return frames\n\n\ndef features(model, frames):\n    import torch\n\n    chw = torch.from_numpy(frames).permute(0, 3, 1, 2)\n    pixels = model.processor.video_processor(chw, return_tensors="pt").pixel_values_videos\n    output = model.get_video_features(pixels.to(model.device, model.dtype))\n    assert output.shape[0] == 1 and output.shape[1] % len(frames) == 0\n    return output\n\n\ndef score(model, system, prefix, option_ids, visual, cached=False):\n    import torch\n\n    lm = model.language_model\n    embed = lm.get_input_embeddings()\n    question = embed(prefix)\n    left = system if visual is None else torch.cat((system, visual), dim=1)\n    if cached:\n        initial = lm(inputs_embeds=left, use_cache=True, return_dict=True)\n        output = lm(inputs_embeds=question, past_key_values=initial.past_key_values, use_cache=True, return_dict=True)\n    else:\n        output = lm(inputs_embeds=torch.cat((left, question), dim=1), use_cache=False, return_dict=True)\n    result = output.logits[0, -1, option_ids].float().detach().cpu()\n    assert torch.isfinite(result).all()\n    return result, left.shape[1] + question.shape[1]\n\n\ndef make_prompt(model, question, choices, order, with_video):\n    import torch\n\n    perm = list(range(len(choices)))\n    if order == "reversed":\n        perm.reverse()\n    letters = LETTERS[:len(choices)]\n    context = "Answer the question about the video." if with_video else "Answer the question."\n    options = "\\n".join(f"({letter}) {choices[i]}" for letter, i in zip(letters, perm, strict=True))\n    text = model.get_prompt(f"{context}\\nQuestion: {question}\\nOptions:\\n{options}\\n"\n                            "Reply with only the letter of the best option.") + "Answer letter:\\n"\n    tokenizer = model.processor.tokenizer\n    ids = tokenizer.encode(text, add_special_tokens=False)\n    tokens = []\n    for letter in letters:\n        suffix = tokenizer.encode(letter, add_special_tokens=False)\n        assert len(suffix) == 1 and tokenizer.encode(text + letter, add_special_tokens=False) == ids + suffix\n        tokens.extend(suffix)\n    assert len(set(tokens)) == len(tokens)\n    return torch.tensor([ids], device=model.device), tokens, perm, sha(text.encode("utf-8"))\n\n\ndef evaluate_pair(model, plan, media, row, paths, preflight=False):\n    import numpy as np\n    import torch\n\n    frames = {branch: decode(paths[branch], media["decoded"][branch]) for branch in "ab"}\n    assert np.array_equal(frames["a"][2:], frames["b"][2:])\n    recent = features(model, frames["a"][2:])\n    a = features(model, frames["a"][:2])\n    b = features(model, frames["b"][:2])\n    assert a.shape == b.shape and a.shape[1] * 2 == recent.shape[1]\n    visuals = {"question_only": None, "recent_four": recent,\n               "history_a": torch.cat((a, recent), dim=1), "history_b": torch.cat((b, recent), dim=1)}\n    lm = model.language_model\n    system = lm.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    results, checks = [], []\n    for order in ORDERS:\n        prompt = make_prompt(model, row["question"], row["options"], order, True)\n        for arm in ARMS:\n            ids, option_ids, perm, prompt_sha = prompt if arm != "question_only" else make_prompt(\n                model, row["question"], row["options"], order, False)\n            started = time.perf_counter()\n            output, count = score(model, system, ids, option_ids, visuals[arm])\n            elapsed = time.perf_counter() - started\n            if preflight:\n                replay, replay_count = score(model, system, ids, option_ids, visuals[arm])\n                torch.testing.assert_close(output, replay, atol=0, rtol=0)\n                assert replay_count == count\n                if arm == "history_a":\n                    cached, _ = score(model, system, ids, option_ids, visuals[arm], cached=True)\n                    torch.testing.assert_close(output, cached, atol=0.025, rtol=0.005)\n                    assert output.argmax().item() == cached.argmax().item(), "Native-cache top option differs"\n                    checks.append(dict(order=order, native_cache_max_abs=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for idx, value in zip(perm, output.tolist(), strict=True):\n                values[idx] = value\n            # Canonical tie-breaking prevents label order from deciding exact ties.\n            pred = LETTERS[max(range(len(values)), key=values.__getitem__)]\n            recent_sha = media["decoded"]["a"]["decoded_rgb_sha256"][-4:] if arm != "question_only" else []\n            earlier_sha = media["decoded"][arm[-1]]["decoded_rgb_sha256"][:2] if arm.startswith("history") else []\n            results.append(dict(pair_id=plan["pair_id"], task=plan["task"], order=order, arm=arm, prediction=pred,\n                                gold_a=row["answer_a"], gold_b=row["answer_b"], scores=values, permutation=perm,\n                                label_token_ids=option_ids, prompt_sha256=prompt_sha, lm_tokens=count,\n                                earlier_frame_sha256=earlier_sha, recent_frame_sha256=recent_sha,\n                                forward_s=elapsed))\n    return results, checks\n\n\ndef summarize(blocks):\n    pairs = []\n    for block in blocks:\n        records = {(r["order"], r["arm"]): r for r in block}\n        pair = dict(pair_id=block[0]["pair_id"], task=block[0]["task"])\n        for order in ORDERS:\n            a, b = (records[order, arm] for arm in ("history_a", "history_b"))\n            assert a["lm_tokens"] == b["lm_tokens"] and a["prompt_sha256"] == b["prompt_sha256"]\n            assert a["recent_frame_sha256"] == b["recent_frame_sha256"]\n            ai, bi = (LETTERS.index(a[f"gold_{branch}"]) for branch in "ab")\n            ma = a["scores"][ai] - a["scores"][bi]\n            mb = b["scores"][ai] - b["scores"][bi]\n            wins = int(a["prediction"] == a["gold_a"]) + int(b["prediction"] == b["gold_b"])\n            reversal = a["prediction"] != b["prediction"]\n            pair[order] = dict(correct_branches=wins, correct_reversal=wins == 2, unchanged=not reversal,\n                               wrong_reversal=reversal and wins != 2, margin_a=ma, margin_b=mb,\n                               directional_change=ma - mb)\n            for arm in ("question_only", "recent_four"):\n                prediction = records[order, arm]["prediction"]\n                pair[order][f"{arm}_correct_branches"] = int(prediction in (a["gold_a"], a["gold_b"]))\n        pair["both_orders_correct"] = all(pair[order]["correct_reversal"] for order in ORDERS)\n        pair["predictions_order_stable"] = {\n            arm: records["canonical", arm]["prediction"] == records["reversed", arm]["prediction"] for arm in ARMS\n        }\n        pairs.append(pair)\n    orders = {order: dict(\n        both_correct=sum(row[order]["correct_reversal"] for row in pairs),\n        correct_branches=sum(row[order]["correct_branches"] for row in pairs),\n        question_only_correct_branches=sum(row[order]["question_only_correct_branches"] for row in pairs),\n        recent_four_correct_branches=sum(row[order]["recent_four_correct_branches"] for row in pairs),\n        wrong_reversals=sum(row[order]["wrong_reversal"] for row in pairs),\n        unchanged=sum(row[order]["unchanged"] for row in pairs),\n        median_directional_change=statistics.median(row[order]["directional_change"] for row in pairs))\n        for order in ORDERS} if pairs else {}\n    robust = sum(row["both_orders_correct"] for row in pairs)\n    numeric_gate = len(pairs) == 9 and robust >= 5 and all(x["median_directional_change"] > 0 for x in orders.values())\n    return dict(pairs=len(pairs), robust_correct_pairs=robust, order_summary=orders, per_pair=pairs,\n                numeric_development_gate=numeric_gate,\n                advancement="pending source-group verification" if numeric_gate else\n                ("stop on this backbone if diagnostic validity holds" if len(pairs) == 9 and robust == 0\n                 else "review diagnostic outcomes before advancing"),\n                source_groups="The pinned annotations do not identify original source groups. Verify independently.")\n\n\ndef package(work):\n    artifact = work / ARTIFACT\n    temporary = work / (ARTIFACT + ".tmp")\n    files = [path for name in ("exports", "checkpoints", "provenance", "pinned_source", "logs")\n             for path in (work / name).rglob("*") if path.is_file() and path.suffix != ".tmp"]\n    files += [path for path in (work / "launch.json", work / "completion.json") if path.exists()]\n    with ZipFile(temporary, "w", ZIP_DEFLATED) as archive:\n        entries = []\n        for path in sorted(files):\n            raw = path.read_bytes()\n            relative = path.relative_to(work).as_posix()\n            archive.writestr(relative, raw)\n            entries.append(dict(path=relative, bytes=len(raw), sha256=sha(raw)))\n        archive.writestr("evidence_manifest.json", json.dumps(entries, indent=2))\n    temporary.replace(artifact)\n    print(f"EVIDENCE_ZIP={artifact}", flush=True)\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    bundle_path = work / "provenance" / "bundle.json"\n    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))\n    selection = cases(bundle)\n    dataset = work / "media" / "data.jsonl"\n    fetch(bundle, "data.jsonl", dataset, bundle["manifest"]["annotation_sha256"])\n    rows = annotation_rows(bundle, dataset.read_bytes())\n    atomic(work / "provenance" / "selected_annotations.json", [rows[plan["pair_id"]] for plan, _, _ in selection])\n    needed = sum(meta["bytes"] for plan, media, _ in selection\n                 for branch, meta in media["decoded"].items()\n                 if not (work / "media" / plan[f"video_{branch}"]).is_file()\n                 or (work / "media" / plan[f"video_{branch}"]).stat().st_size != meta["bytes"])\n    assert shutil.disk_usage(work).free >= needed + 256 * 1024**2, "Free data-disk space before running"\n    locations = {}\n    print(f"MEDIA_PLAN 18 pinned clips, up to {needed / 1024**2:.1f} MiB not present", flush=True)\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        pending = []\n        for plan, media, _ in selection:\n            locations[plan["pair_id"]] = {}\n            for branch, meta in media["decoded"].items():\n                relative = plan[f"video_{branch}"]\n                dest = work / "media" / relative\n                reuse = args.media_reuse / Path(relative).relative_to("videos")\n                pending.append(pool.submit(fetch, bundle, relative, dest, meta["sha256"], meta["bytes"], reuse))\n                locations[plan["pair_id"]][branch] = dest\n        for download in pending:\n            download.result()\n    settings = dict(bundle_sha256=digest(bundle_path), runner_sha256=digest(Path(__file__)),\n                    model_name=args.model.name, torch_version=torch.__version__,\n                    gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,\n                    driver=torch.cuda.driver_version() if hasattr(torch.cuda, "driver_version") else None)\n    identity = sha(json.dumps(settings, sort_keys=True).encode())\n    checkpoint_dir = work / "checkpoints"\n    checkpoint_dir.mkdir(exist_ok=True)\n    atomic(work / "exports" / "run_config.json", settings)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    blocks = []\n    with torch.inference_mode():\n        for index, (plan, media, _) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            checkpoint = checkpoint_dir / f"{pair_id}.json"\n            if index and checkpoint.is_file():\n                saved = json.loads(checkpoint.read_text(encoding="utf-8"))\n                assert saved["identity"] == identity and saved["pair_id"] == pair_id, "Use a new WORK for changed settings"\n                records = saved["records"]\n                assert len(records) == 8 and {(r["order"], r["arm"]) for r in records} == {\n                    (order, arm) for order in ORDERS for arm in ARMS}\n                assert all(r["pair_id"] == pair_id for r in records)\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n            else:\n                print(f"PAIR_RUNNING {index + 1}/9 {pair_id}", flush=True)\n                records, checks = evaluate_pair(model, plan, media, rows[pair_id], locations[pair_id], preflight=index == 0)\n                if index == 0:\n                    if checkpoint.exists():\n                        assert json.loads(checkpoint.read_text(encoding="utf-8"))["identity"] == identity, (\n                            "Use a new WORK for changed settings")\n                    atomic(work / "exports" / "preflight.json", dict(status="pass", pair_id=pair_id, checks=checks))\n                    print("TWINPAST_PREFLIGHT_PASS", flush=True)\n                atomic(checkpoint, dict(identity=identity, pair_id=pair_id, records=records))\n                print(f"PAIR_DONE {index + 1}/9 {pair_id}", flush=True)\n            blocks.append(records)\n    summary = summarize(blocks)\n    atomic(work / "exports" / "summary.json", summary)\n    fields = ("pair_id", "task", "order", "arm", "prediction", "gold_a", "gold_b", "scores", "permutation",\n              "label_token_ids", "prompt_sha256", "lm_tokens", "earlier_frame_sha256", "recent_frame_sha256", "forward_s")\n    with (work / "exports" / "scores.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value) if isinstance(value, list) else value for key, value in row.items()}\n                         for block in blocks for row in block)\n    atomic(work / "completion.json", dict(status="complete", pairs=len(blocks), scores=sum(map(len, blocks))))\n    print(f"TWINPAST_DIAGNOSTIC_COMPLETE pairs={len(blocks)} scores={sum(map(len, blocks))}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--work", required=True, type=Path)\n    parser.add_argument("--model", required=True, type=Path)\n    parser.add_argument("--media-reuse", type=Path, default=Path("/root/autodl-tmp/twinpast-development"))\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/history_only_runner.py': '"""Test whether earlier evidence alone supports the frozen TwinPast answers."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nfrom pathlib import Path\nimport time\n\nimport twinpast_base as base\n\nARTIFACT = "HERMES_twinpast_history_only_evidence.zip"\nDENSE_COUNT = 6\n\n\ndef dense_indices(seconds, intervals, earlier, cutoff):\n    """Keep the two original frames and add four spread across valid native timestamps."""\n    import numpy as np\n\n    mask = np.zeros(len(seconds), dtype=bool)\n    for start, stop in intervals:\n        mask |= (seconds >= start - 1e-6) & (seconds <= stop + 1e-6)\n    eligible = np.flatnonzero(mask & (seconds < cutoff)).tolist()\n    assert sorted(set(earlier)) == earlier and len(earlier) == 2\n    assert set(earlier) <= set(eligible), "Frozen history frame outside eligible annotated evidence"\n    if len(eligible) < DENSE_COUNT:\n        raise ValueError(f"Only {len(eligible)} distinct annotated frames before recent context")\n    chosen = set(earlier)\n    for k in range(DENSE_COUNT - len(earlier)):\n        target = k * (len(eligible) - 1) / (DENSE_COUNT - len(earlier) - 1)\n        pos = min((i for i, ix in enumerate(eligible) if ix not in chosen), key=lambda i: abs(i - target))\n        chosen.add(eligible[pos])\n    return sorted(chosen)\n\n\ndef prepare_dense(paths, annotated, media):\n    import numpy as np\n    from decord import VideoReader\n\n    result = {}\n    cutoff = min(annotated["t_query"], *(media["decoded"][b]["frame_start_s"][2] for b in "ab"))\n    for branch in "ab":\n        reader = VideoReader(str(paths[branch]), num_threads=1)\n        seconds = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n        seconds -= seconds[0, 0]\n        seconds = seconds[:, 0]\n        indices = dense_indices(seconds, annotated["evidence_intervals"][branch.upper()],\n                                media["decoded"][branch]["source_frames"][:2], cutoff)\n        frames = reader.get_batch(indices).asnumpy()\n        result[branch] = dict(frames=frames, indices=indices, seconds=seconds[indices].tolist(),\n                              cutoff_s=cutoff, hashes=[base.sha(frame.tobytes()) for frame in frames])\n    return result\n\n\ndef contact_sheet(path, pair_id, condition, windows):\n    from PIL import Image, ImageDraw\n\n    cols = len(windows["a"]["frames"])\n    canvas = Image.new("RGB", (320 * cols, 450), "white")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((8, 5), f"{pair_id} | {condition} | Native frames resized to fit", fill="black")\n    for row, branch in enumerate("ab"):\n        for col, (frame, seconds) in enumerate(zip(windows[branch]["frames"], windows[branch]["seconds"], strict=True)):\n            x, y = col * 320, 23 + row * 210\n            picture = Image.fromarray(frame)\n            picture.thumbnail((320, 180))\n            canvas.paste(picture, (x, y))\n            draw.text((x + 4, y + 182), f"{branch.upper()} | {seconds:.3f} s", fill="black")\n    path.parent.mkdir(exist_ok=True)\n    canvas.save(path, quality=90)\n\n\ndef evaluate(model, annotated, windows, reference, preflight=False):\n    import torch\n\n    system = model.language_model.get_input_embeddings()(\n        torch.as_tensor(model.init_prompt_ids, device=model.device))\n    visual = {branch: base.features(model, window["frames"]) for branch, window in windows.items()}\n    assert visual["a"].shape == visual["b"].shape\n    rows, checks = [], []\n    for order in base.ORDERS:\n        ids, option_ids, perm, prompt_sha = base.make_prompt(\n            model, annotated["question"], annotated["options"], order, True)\n        for branch in "ab":\n            old = reference[order, branch]\n            assert old["prompt_sha256"] == prompt_sha and old["label_token_ids"] == option_ids\n            start = time.perf_counter()\n            output, length = base.score(model, system, ids, option_ids, visual[branch])\n            elapsed = time.perf_counter() - start\n            if preflight:\n                repeat, _ = base.score(model, system, ids, option_ids, visual[branch])\n                torch.testing.assert_close(output, repeat, rtol=0, atol=0)\n                if branch == "a":\n                    cached, _ = base.score(model, system, ids, option_ids, visual[branch], cached=True)\n                    torch.testing.assert_close(output, cached, rtol=0.005, atol=0.025)\n                    assert output.argmax().item() == cached.argmax().item()\n                    checks.append(dict(order=order, max_native_cache_diff=float((output - cached).abs().max())))\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, output.tolist(), strict=True):\n                values[i] = value\n            prediction = base.LETTERS[max(range(len(values)), key=values.__getitem__)]\n            gold = annotated[f"answer_{branch}"]\n            window = windows[branch]\n            rows.append(dict(order=order, branch=branch, prediction=prediction, gold=gold, scores=values,\n                             prompt_sha256=prompt_sha, lm_tokens=length, frame_indices=window["indices"],\n                             frame_seconds=window["seconds"], frame_hashes=window["hashes"],\n                             prior_prediction=old["prediction"], prior_scores=old["scores"], forward_s=elapsed))\n        assert rows[-1]["lm_tokens"] == rows[-2]["lm_tokens"]\n    return rows, checks\n\n\ndef outcome(rows, annotated):\n    wins = {}\n    ai, bi = (base.LETTERS.index(annotated[f"answer_{b}"]) for b in "ab")\n    for order in base.ORDERS:\n        a, b = (next(r for r in rows if r["order"] == order and r["branch"] == branch) for branch in "ab")\n        margin = lambda x: x[ai] - x[bi]\n        wins[order] = dict(branches_correct=int(a["prediction"] == a["gold"]) + int(b["prediction"] == b["gold"]),\n                           directional_change=margin(a["scores"]) - margin(b["scores"]))\n    return dict(orders=wins, robust=all(row["branches_correct"] == 2 for row in wins.values()))\n\n\ndef acquire(args, bundle, selection):\n    paths = {}\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        downloads = []\n        for plan, media, _ in selection:\n            pair_paths = paths[plan["pair_id"]] = {}\n            for branch, check in media["decoded"].items():\n                relative = Path(plan[f"video_{branch}"])\n                reused = args.previous / "media" / relative\n                if reused.is_file() and base.digest(reused) == check["sha256"]:\n                    pair_paths[branch] = reused\n                    print(f"MEDIA_REUSED {relative.as_posix()}", flush=True)\n                else:\n                    dest = args.work / "media" / relative\n                    downloads.append(pool.submit(base.fetch, bundle, relative.as_posix(), dest, check["sha256"],\n                                                 check["bytes"], args.media_reuse / relative.relative_to("videos")))\n                    pair_paths[branch] = dest\n        for download in downloads:\n            download.result()\n    return paths\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    prior = json.loads((provenance / "prior.json").read_bytes())\n    selection = base.cases(bundle)\n    annotations = {r["pair_id"]: r for r in prior["annotations"]}\n    assert annotations.keys() == {plan["pair_id"] for plan, _, _ in selection}\n    paths = acquire(args, bundle, selection)\n    print("LOADING_PINNED_MODEL", args.model.name, flush=True)\n    model, _ = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda"\n    config = dict(model_revision=args.model.name, gpu=torch.cuda.get_device_name(0), torch=torch.__version__,\n                  cuda=torch.version.cuda, provenance_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir())})\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    base.atomic(exports / "run_config.json", config)\n    reports = []\n    with torch.inference_mode():\n        for index, (plan, media, screened) in enumerate(selection):\n            pair_id = plan["pair_id"]\n            annotated = annotations[pair_id]\n            assert all(annotated[f"answer_{b}"] == screened[f"observed_option_{b}"] for b in "ab")\n            old = {(r["order"], r["arm"][-1]): r for r in prior["pairs"][pair_id]}\n            checkpoint = checkpoints / f"{pair_id}.json"\n            expected = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if expected is not None:\n                assert expected["identity"] == ident and expected["pair_id"] == pair_id\n            if index and expected is not None:\n                reports.append(expected["summary"])\n                print(f"PAIR_REUSED {index + 1}/9 {pair_id}", flush=True)\n                continue\n            two = {}\n            for branch in "ab":\n                record = media["decoded"][branch]\n                assert all(r["earlier_frame_sha256"] == record["decoded_rgb_sha256"][:2] for (_, b), r in old.items() if b == branch)\n                frames = base.decode(paths[pair_id][branch], record)\n                two[branch] = dict(frames=frames[:2], indices=record["source_frames"][:2],\n                                   seconds=record["frame_start_s"][:2], hashes=record["decoded_rgb_sha256"][:2])\n            rows, checks = evaluate(model, annotated, two, old, preflight=index == 0)\n            if index == 0:\n                base.atomic(exports / "preflight.json", dict(status="pass", checks=checks))\n                print("TWINPAST_HISTORY_ONLY_PREFLIGHT_PASS", flush=True)\n            comparison = outcome(rows, annotated)\n            summary = dict(pair_id=pair_id, task=plan["task"], two_history_only=comparison,\n                           prior_history_with_recent_robust=all(r["prediction"] == r[f"gold_{b}"] for (_, b), r in old.items()))\n            dense_rows = []\n            if plan["task"] == "HSR":\n                contact_sheet(exports / "review" / f"{pair_id}_two.jpg", pair_id, "Two earlier frames", two)\n                if not comparison["robust"]:\n                    try:\n                        dense = prepare_dense(paths[pair_id], annotated, media)\n                    except ValueError as exc:\n                        summary["dense_skip"] = str(exc)\n                    else:\n                        dense_rows, _ = evaluate(model, annotated, dense, old)\n                        summary["dense_history_only"] = outcome(dense_rows, annotated)\n                        contact_sheet(exports / "review" / f"{pair_id}_dense.jpg", pair_id, "Six earlier frames", dense)\n            base.atomic(checkpoint, dict(pair_id=pair_id, identity=ident, summary=summary,\n                                        history_two=rows, history_dense=dense_rows))\n            reports.append(summary)\n            print(f"PAIR_DONE {index + 1}/9 {pair_id} two_order_robust={comparison[\'robust\']}", flush=True)\n    dense = [item["dense_history_only"] for item in reports if "dense_history_only" in item]\n    summary = dict(pairs=len(reports), two_history_only_robust=sum(r["two_history_only"]["robust"] for r in reports),\n                   restored_from_original=sum(r["two_history_only"]["robust"] and not r["prior_history_with_recent_robust"]\n                                              for r in reports),\n                   lost_from_original=sum(not r["two_history_only"]["robust"] and r["prior_history_with_recent_robust"]\n                                          for r in reports),\n                   dense_pairs=len(dense), dense_robust=sum(r["robust"] for r in dense), per_pair=reports)\n    base.atomic(exports / "summary.json", summary)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(reports), dense_pairs=len(dense)))\n    print("TWINPAST_HISTORY_ONLY_COMPLETE", json.dumps({k: v for k, v in summary.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/backbone_runner.py': '"""Replay the frozen history-only windows on the larger same-family backbone."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport history_only_runner as history\n\nbase = history.base\nARTIFACT = "HERMES_twinpast_backbone_evidence.zip"\nMODEL_REVISION = "0d50680527681998e456c7b78950205bedd8a068"\n\n\ndef reproduce_windows(paths, records):\n    windows = {}\n    for branch in "ab":\n        left, right = (next(r for r in records if r["branch"] == branch and r["order"] == order)\n                       for order in base.ORDERS)\n        assert all(left[k] == right[k] for k in ("frame_indices", "frame_seconds", "frame_hashes"))\n        frames = base.decode(paths[branch], dict(source_frames=left["frame_indices"],\n                                                 decoded_rgb_sha256=left["frame_hashes"]))\n        windows[branch] = dict(frames=frames, indices=left["frame_indices"],\n                               seconds=left["frame_seconds"], hashes=left["frame_hashes"])\n    return windows\n\n\ndef run_pair(model, paths, case, smoke=False):\n    windows = reproduce_windows(paths, case["reference_rows"])\n    original = {(r["order"], r["arm"][-1]): r for r in case["original_prompt_rows"]}\n    reference = {}\n    for row in case["reference_rows"]:\n        key = row["order"], row["branch"]\n        assert row["prompt_sha256"] == original[key]["prompt_sha256"]\n        reference[key] = {**row, "label_token_ids": original[key]["label_token_ids"]}\n    rows, checks = history.evaluate(model, case["annotation"], windows, reference, preflight=smoke)\n    prior = history.outcome(case["reference_rows"], case["annotation"])\n    current = history.outcome(rows, case["annotation"])\n    return dict(pair_id=case["annotation"]["pair_id"], cohort=case["cohort"], prior=prior,\n                current=current, preflight=checks, rows=rows)\n\n\ndef summarize(blocks):\n    states = [x for x in blocks if x["cohort"] == "historical_state"]\n    controls = [x for x in blocks if x["cohort"] == "positive_control"]\n    robust_states = sum(x["current"]["robust"] for x in states)\n    robust_controls = sum(x["current"]["robust"] for x in controls)\n    return dict(pairs=len(blocks), scores=sum(len(x["rows"]) for x in blocks),\n                historical_state_robust=robust_states, historical_state_total=len(states),\n                positive_control_robust=robust_controls, positive_control_total=len(controls),\n                development_criterion_met=len(states) == 4 and len(controls) == 3\n                and robust_states >= 2 and robust_controls >= 2,\n                per_pair=[{k: v for k, v in x.items() if k not in {"preflight", "rows"}} for x in blocks])\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    bundle = json.loads((provenance / "bundle.json").read_bytes())\n    frozen = json.loads((provenance / "backbone_cases.json").read_bytes())\n    cases = {c["annotation"]["pair_id"]: c for c in frozen["cases"]}\n    assert len(cases) == 7 and frozen["model_revision"] == MODEL_REVISION == args.model.name\n    selection = [pair for pair in base.cases(bundle) if pair[0]["pair_id"] in cases]\n    assert {p["pair_id"] for p, _, _ in selection} == cases.keys()\n    # Start with the six-frame historical-state case, which exercises the longest input.\n    assert cases[selection[0][0]["pair_id"]]["cohort"] == "historical_state"\n    paths = history.acquire(SimpleNamespace(work=work, previous=args.previous, media_reuse=args.media_reuse),\n                            bundle, selection)\n    print("LOADING_PINNED_7B", args.model.name, flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.device.type == "cuda" and model.dtype == torch.float16\n    assert {str(p.device) for p in model.parameters()} == {"cuda:0"}, "7B must fit fully on GPU 0"\n    config = dict(model_revision=MODEL_REVISION, gpu=torch.cuda.get_device_name(0),\n                  torch=torch.__version__, cuda=torch.version.cuda, dtype=str(model.dtype),\n                  video_processor=processor.video_processor.to_dict(),\n                  runtime_source_hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()},\n                  source_history_only_archive=frozen["history_only_archive_sha256"])\n    ident = base.sha(json.dumps(config, sort_keys=True).encode())\n    base.atomic(work / "exports" / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, (plan, _, _) in enumerate(selection):\n            pair = plan["pair_id"]\n            checkpoint = checkpoints / f"{pair}.json"\n            saved = json.loads(checkpoint.read_bytes()) if checkpoint.is_file() else None\n            if saved is not None:\n                assert saved["identity"] == ident and saved["block"]["pair_id"] == pair\n            if i and saved is not None:\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i + 1}/7 {pair}", flush=True)\n                continue\n            print(f"PAIR_RUNNING {i + 1}/7 {pair}", flush=True)\n            torch.cuda.reset_peak_memory_stats()\n            block = run_pair(model, paths[pair], cases[pair], smoke=i == 0)\n            block["peak_gpu_allocated_bytes"] = torch.cuda.max_memory_allocated()\n            if i == 0:\n                base.atomic(work / "exports" / "preflight.json",\n                            dict(status="pass", pair_id=pair, checks=block["preflight"],\n                                 peak_gpu_allocated_bytes=block["peak_gpu_allocated_bytes"]))\n                print("TWINPAST_BACKBONE_PREFLIGHT_PASS", flush=True)\n            base.atomic(checkpoint, dict(identity=ident, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i + 1}/7 {pair} robust={block[\'current\'][\'robust\']}", flush=True)\n    report = summarize(blocks)\n    assert report["pairs"] == 7 and report["scores"] == 28\n    base.atomic(work / "exports" / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=7, scores=28))\n    print("TWINPAST_BACKBONE_COMPLETE", json.dumps({k: v for k, v in report.items() if k != "per_pair"}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'provenance/automatic_runner.py': '"""Run a frozen, annotation-blind retrieval and answer-correction screen."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport backbone_runner as backbone\n\nbase = backbone.base\nARTIFACT = "HERMES_twinpast_automatic_evidence.zip"\nMAX_CANDIDATES = 13\nMAX_HISTORY = 6\nMIN_LIFT = 0.5\nMETHODS = ("recent", "latest", "uniform", "relevance", "margin_gate", "history_control_gate")\n\n\ndef spread(count, budget):\n    if count <= budget:\n        return list(range(count))\n    return [i * (count - 1) // (budget - 1) for i in range(budget)]\n\n\ndef sample_prefix(path, query):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    grid = list(range(0, math.ceil(query), 2))\n    assert len(grid) >= 8 and grid[-1] < query and grid[-1] <= times[-1, 1] + 1e-6\n    older_grid, recent_grid = grid[:-4], grid[-4:]\n    picks = [older_grid[i] for i in spread(len(older_grid), MAX_CANDIDATES)] + recent_grid\n    indices = np.searchsorted(times[:, 0], picks, side="right") - 1\n    sampled_s = times[indices, 0]\n    assert indices[0] >= 0 and np.all(np.diff(indices) > 0)\n    assert np.all(sampled_s <= picks) and sampled_s[-1] < query\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=picks, indices=indices.tolist(), timestamps_s=sampled_s.tolist(),\n                        hashes=[base.sha(f.tobytes()) for f in frames])\n\n\ndef encode_frames(model, frames):\n    """Vision encoding is frame-local. Limit the encoder batch to four frames."""\n    bank = []\n    for start in range(0, len(frames), 4):\n        batch = frames[start:start + 4]\n        encoded = base.features(model, batch)\n        bank.extend(encoded.split(encoded.shape[1] // len(batch), dim=1))\n    assert len(bank) == len(frames)\n    return bank\n\n\ndef best(scores):\n    return max(range(len(scores)), key=scores.__getitem__)\n\n\ndef answer_policies(recent, direct, uniform, same_history):\n    initial, alternative = best(recent), best(direct)\n    margin = direct[alternative] - direct[initial]\n    lift = margin - (recent[alternative] - recent[initial])\n    specificity = margin - (uniform[alternative] - uniform[initial])\n    accept = alternative != initial and lift >= MIN_LIFT\n    controlled = accept and not same_history and specificity >= MIN_LIFT\n    return dict(relevance=alternative, margin_gate=alternative if accept else initial,\n                history_control_gate=alternative if controlled else initial,\n                scores=dict(history_margin=margin, lift_from_recent=lift, lift_from_uniform=specificity))\n\n\ndef branch_policy(model, path, query, question, options):\n    """Infer for one branch with no evidence labels, gold, or counterpart data."""\n    import torch\n\n    started = time.perf_counter()\n    frames, sample = sample_prefix(path, query)\n    bank = encode_frames(model, frames)\n    history_count = len(bank) - 4\n    size = min(MAX_HISTORY, history_count)\n    recent = list(range(history_count, len(bank)))\n    uniform = spread(history_count, size)\n    latest = list(range(history_count - size, history_count))\n    embedding = model.language_model.get_input_embeddings()\n    system = embedding(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    calls = []\n    logits_cache = {}\n\n    def score(prompt, window, phase, order):\n        ids, option_ids, perm, digest = prompt\n        key = digest, tuple(window)\n        if key not in logits_cache:\n            visual = torch.cat([bank[i] for i in window], dim=1)\n            tick = time.perf_counter()\n            logits, length = base.score(model, system, ids, option_ids, visual)\n            values = [0.0] * len(perm)\n            for i, value in zip(perm, logits.tolist(), strict=True):\n                values[i] = value\n            logits_cache[key] = values\n            calls.append(dict(phase=phase, order=order, slots=window, forward_s=time.perf_counter()-tick,\n                              visual_tokens=visual.shape[1], lm_tokens=length, prompt_sha256=digest))\n        return logits_cache[key]\n\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {o: base.make_prompt(model, probe, ["Yes", "No"], o, True) for o in base.ORDERS}\n    candidates = []\n    relevance = [-math.inf] * history_count\n    for slot in range(history_count - 1):\n        window = [slot, slot + 1]\n        probe_scores = {o: score(prompts[o], window, "retrieval", o) for o in base.ORDERS}\n        average = sum(v[0] - v[1] for v in probe_scores.values()) / len(probe_scores)\n        candidates.append(dict(slots=window, yes_no=probe_scores, relevance=average))\n        for i in window:\n            relevance[i] = max(relevance[i], average)\n    ranked = sorted(sorted(range(history_count), key=lambda i: (-relevance[i], i))[:size])\n    assert len(set(ranked)) == len(uniform) == len(latest) == size\n    selection = {"recent": recent, "latest": latest + recent,\n                 "uniform": uniform + recent, "relevance": ranked + recent}\n    answers = {}\n    for order in base.ORDERS:\n        prompt = base.make_prompt(model, question, options, order, True)\n        vectors = {arm: score(prompt, slots, arm, order) for arm, slots in selection.items()}\n        decision = answer_policies(vectors["recent"], vectors["relevance"], vectors["uniform"], ranked == uniform)\n        choices = {arm: best(values) for arm, values in vectors.items()}\n        choices.update({k: decision[k] for k in ("margin_gate", "history_control_gate")})\n        answers[order] = dict(logits=vectors, predictions={arm: base.LETTERS[i] for arm, i in choices.items()},\n                              margins=decision["scores"], prompt_sha256=prompt[3])\n    return dict(sample=sample, history_budget=size, candidates=candidates, selection=selection, answers=answers,\n                calls=calls, wall_s=time.perf_counter()-started)\n\n\ndef summarize(blocks, targets):\n    report = {}\n    opened = {arm: 0 for arm in ("margin_gate", "history_control_gate")}\n    gate_wins, gate_losses = 0, 0\n    for arm in METHODS:\n        robust, wins, losses = [], 0, 0\n        for block in blocks:\n            pair_id = block["pair_id"]\n            success = True\n            for order in base.ORDERS:\n                for branch in "ab":\n                    values = block["branches"][branch]["answers"][order]["predictions"]\n                    gold = targets[pair_id][f"answer_{branch}"]\n                    correct = values[arm] == gold\n                    reference = values["recent"] == gold\n                    wins += correct and not reference\n                    losses += reference and not correct\n                    success = success and correct\n                    if arm in opened:\n                        opened[arm] += values[arm] != values["recent"]\n                    if arm == "history_control_gate":\n                        direct_correct = values["relevance"] == gold\n                        gate_wins += correct and not direct_correct\n                        gate_losses += direct_correct and not correct\n            if success:\n                robust.append(pair_id)\n        report[arm] = dict(robust_pairs=robust, recoveries_vs_recent=wins, regressions_vs_recent=losses)\n    return dict(pairs=len(blocks), branch_order_answers=len(blocks)*4, methods=report, overrides=opened,\n                controlled_gate_vs_same_frames_direct=dict(recoveries=gate_wins, regressions=gate_losses),\n                total_forward_calls=sum(len(v["calls"]) for b in blocks for v in b["branches"].values()))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "automatic_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    cases = {c["pair_id"]: c for c in spec["cases"]}\n    assert len(cases) == 7\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, torch=torch.__version__, cuda=torch.version.cuda,\n                  gpu=torch.cuda.get_device_name(0), min_lift=MIN_LIFT, max_candidates=MAX_CANDIDATES,\n                  max_history=MAX_HISTORY, source_archive_sha256=spec["fixed_recent_archive_sha256"],\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    exports = work / "exports"\n    base.atomic(exports / "run_config.json", config)\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(spec["cases"]):\n            pair_id = case["pair_id"]\n            checkpoint = checkpoints / f"{pair_id}.json"\n            if checkpoint.is_file():\n                saved = json.loads(checkpoint.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair_id\n                blocks.append(saved["block"])\n                print(f"PAIR_REUSED {i+1}/7 {pair_id}", flush=True)\n                continue\n            torch.cuda.reset_peak_memory_stats()\n            branches = {b: branch_policy(model, paths[pair_id][b], case["query_s"], case["question"], case["options"])\n                        for b in "ab"}\n            left, right = branches["a"], branches["b"]\n            assert left["sample"]["grid_s"] == right["sample"]["grid_s"]\n            assert left["sample"]["hashes"][-4:] == right["sample"]["hashes"][-4:] == case["recent_hashes"]\n            assert left["history_budget"] == right["history_budget"]\n            for order in base.ORDERS:\n                assert left["answers"][order]["predictions"]["recent"] == right["answers"][order]["predictions"]["recent"]\n            block = dict(pair_id=pair_id, branches=branches,\n                         peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n            base.atomic(checkpoint, dict(identity=identity, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i+1}/7 {pair_id}", flush=True)\n    # Ground truth is loaded only after all branch predictions have been written.\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    assert targets.keys() == cases.keys()\n    report = summarize(blocks, targets)\n    coverage = []\n    for block in blocks:\n        pair_id = block["pair_id"]\n        for branch, trace in block["branches"].items():\n            intervals = targets[pair_id]["evidence_intervals"][branch.upper()]\n            timestamps = trace["sample"]["timestamps_s"]\n            inside = [i for i, t in enumerate(timestamps[:-4]) if any(lo <= t <= hi for lo, hi in intervals)]\n            coverage.append(dict(pair_id=pair_id, branch=branch, candidate_slots_in_evidence=inside,\n                                 selected_slots_in_evidence={name: sorted(set(inside) & set(trace["selection"][name]))\n                                                             for name in ("latest", "uniform", "relevance")}))\n    base.atomic(exports / "postrun_evidence_overlap.json", coverage)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_AUTOMATIC_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/temporal_runner.py': '"""Evaluate endpoint-preserving retrieval on a new TwinPast development screen."""\nimport argparse\nfrom collections import Counter\nfrom itertools import combinations\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport automatic_runner as auto\n\nbase = auto.base\nbackbone = auto.backbone\nARTIFACT = "HERMES_twinpast_temporal_windows_evidence.zip"\nMETHODS = ("recent", "uniform", "individual", "endpoints")\nWINDOW_SPAN = 3\nWINDOW_CAP = 12\n\n\ndef decode_grid(path, grid):\n    import numpy as np\n    from decord import VideoReader\n\n    reader = VideoReader(str(path), num_threads=1)\n    times = np.asarray(reader.get_frame_timestamp(list(range(len(reader)))), dtype=np.float64)\n    times -= times[0, 0]\n    if grid is None:\n        # Source-screening frames never enter the policy.\n        grid = range(0, math.ceil(times[-1, 1]), 2)\n    grid = np.asarray(list(grid))\n    if grid[-1] > times[-1, 1] + 1e-6:\n        raise ValueError("Clip ends before the frozen grid")\n    indices = np.searchsorted(times[:, 0], grid, side="right") - 1\n    if indices[0] < 0 or not np.all(np.diff(indices) > 0):\n        raise ValueError("Clip does not contain distinct frames on the frozen grid")\n    frames = reader.get_batch(indices.tolist()).asnumpy()\n    return frames, dict(grid_s=grid.tolist(), indices=indices.tolist(), timestamps_s=times[indices, 0].tolist(),\n                        hashes=[base.sha(frame.tobytes()) for frame in frames])\n\n\ndef fingerprints(path):\n    import numpy as np\n    from PIL import Image\n\n    frames, trace = decode_grid(path, None)\n    signatures = []\n    for frame, sha in zip(frames, trace["hashes"], strict=True):\n        image = Image.fromarray(frame)\n        gray = np.asarray(image.convert("L").resize((9, 8)))\n        bits = int.from_bytes(np.packbits((gray[:, 1:] > gray[:, :-1]).reshape(-1)).tobytes(), "big")\n        rgb = np.asarray(image.resize((4, 4)), dtype=int).reshape(-1)\n        if gray.std() >= 8:\n            signatures.append((sha, bits, rgb))\n    return signatures\n\n\ndef shared_frames(a, b):\n    """Flag repeat footage on at least three distinct sampled frame matches."""\n    import numpy as np\n\n    matches = set()\n    for sha_a, hash_a, rgb_a in a:\n        for sha_b, hash_b, rgb_b in b:\n            if sha_a == sha_b or ((hash_a ^ hash_b).bit_count() <= 4 and np.abs(rgb_a - rgb_b).mean() <= 6):\n                matches.add(sha_b)\n                break\n    return len(matches)\n\n\ndef screen_pairs(cases, excluded, paths, max_pairs):\n    signature_bank = {pair: {b: fingerprints(paths[pair][b]) for b in "ab"} for pair in excluded}\n    decisions, accepted = [], []\n    for case in cases:\n        pair = case["pair_id"]\n        if len(accepted) == max_pairs:\n            break\n        grid = list(range(math.ceil(case["query_s"])))\n        try:\n            recent = {b: decode_grid(paths[pair][b], grid[-4:])[1]["hashes"] for b in "ab"}\n        except ValueError as exc:\n            decisions.append(dict(pair_id=pair, disposition="invalid_grid", detail=str(exc)))\n            continue\n        if recent["a"] != recent["b"]:\n            decisions.append(dict(pair_id=pair, disposition="recent_context_mismatch"))\n            continue\n        candidate = {b: fingerprints(paths[pair][b]) for b in "ab"}\n        overlaps = []\n        for reference, signature in signature_bank.items():\n            matches = max(shared_frames(candidate[b], signature[c]) for b in "ab" for c in "ab")\n            if matches >= 3:\n                overlaps.append(dict(reference=reference, sampled_matches=matches))\n        if overlaps:\n            decisions.append(dict(pair_id=pair, disposition="possible_media_overlap", matches=overlaps))\n            continue\n        signature_bank[pair] = candidate\n        accepted.append(case)\n        decisions.append(dict(pair_id=pair, disposition="accepted", recent_hashes=recent["a"]))\n    return accepted, decisions\n\n\ndef select_windows(windows):\n    order = sorted(windows, key=lambda row: row["slots"])\n    compatible = (group for group in combinations(order, 3)\n                  if all(left["slots"][-1] < right["slots"][0] for left, right in zip(group, group[1:])))\n    best = max(compatible, key=lambda group: sum(r["relevance"] for r in group))\n    return [i for r in best for i in r["slots"]]\n\n\ndef branch_policy(model, path, case):\n    import torch\n\n    tick = time.perf_counter()\n    frames, sample = decode_grid(path, range(math.ceil(case["query_s"])))\n    assert sample["timestamps_s"][-1] < case["query_s"]\n    bank = auto.encode_frames(model, frames)\n    num_history = len(frames) - 4\n    recent = list(range(num_history, len(frames)))\n    system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    log, cache = [], {}\n\n    def score(prompt, slots, phase, order):\n        ids, option_ids, permutation, prompt_hash = prompt\n        key = (prompt_hash, tuple(slots))\n        if key not in cache:\n            visual = torch.cat([bank[i] for i in slots], dim=1)\n            started = time.perf_counter()\n            values, tokens = base.score(model, system, ids, option_ids, visual)\n            canonical = [0.0] * len(permutation)\n            for i, value in zip(permutation, values.tolist(), strict=True):\n                canonical[i] = value\n            cache[key] = canonical\n            log.append(dict(phase=phase, order=order, slots=slots, forward_s=time.perf_counter()-started,\n                            lm_tokens=tokens, visual_tokens=visual.shape[1]))\n        return cache[key]\n\n    question = case["question"]\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + question)\n    prompts = {order: base.make_prompt(model, probe, ["Yes", "No"], order, True) for order in base.ORDERS}\n    windows, individual_scores = [], {}\n    for start in auto.spread(num_history - WINDOW_SPAN, WINDOW_CAP):\n        endpoints = [start, start + WINDOW_SPAN]\n        vectors = {o: score(prompts[o], endpoints, "retrieval", o) for o in base.ORDERS}\n        value = sum(v[0] - v[1] for v in vectors.values()) / len(vectors)\n        windows.append(dict(slots=endpoints, relevance=value, yes_no=vectors))\n        for i in endpoints:\n            individual_scores[i] = max(value, individual_scores.get(i, -math.inf))\n    selection = dict(recent=recent, uniform=auto.spread(num_history, 6) + recent,\n                     individual=sorted(sorted(individual_scores, key=lambda i: (-individual_scores[i], i))[:6]) + recent,\n                     endpoints=select_windows(windows) + recent)\n    assert len(selection["recent"]) == 4 and all(len(selection[m]) == 10 for m in METHODS[1:])\n    assert all(slots == sorted(set(slots)) for slots in selection.values())\n    answers = {}\n    for o in base.ORDERS:\n        prompt = base.make_prompt(model, question, case["options"], o, True)\n        vectors = {m: score(prompt, selection[m], m, o) for m in METHODS}\n        answers[o] = dict(logits=vectors, predictions={m: base.LETTERS[auto.best(v)] for m, v in vectors.items()})\n    return dict(sample=sample, windows=windows, selection=selection, answers=answers,\n                calls=log, wall_s=time.perf_counter()-tick)\n\n\ndef evaluate(blocks, targets):\n    methods = {}\n    for method in METHODS:\n        pairs, correct = [], 0\n        for block in blocks:\n            p = block["pair_id"]\n            wins = [block["branches"][b]["answers"][o]["predictions"][method] == targets[p][f"answer_{b}"]\n                    for b in "ab" for o in base.ORDERS]\n            correct += sum(wins)\n            if all(wins):\n                pairs.append(p)\n        methods[method] = dict(robust_pairs=pairs, correct_branch_orders=correct)\n    endpoints = set(methods["endpoints"]["robust_pairs"])\n    matched = {m: dict(recoveries=sorted(endpoints - set(methods[m]["robust_pairs"])),\n                       regressions=sorted(set(methods[m]["robust_pairs"]) - endpoints))\n               for m in ("uniform", "individual")}\n    calls = Counter(call["phase"] for p in blocks for b in p["branches"].values() for call in b["calls"])\n    return dict(pairs=len(blocks), methods=methods, matched_pair_changes=matched,\n                endpoints_more_robust_than_both_dense_baselines=len(methods["endpoints"]["robust_pairs"]) >\n                max(len(methods[m]["robust_pairs"]) for m in ("uniform", "individual")),\n                forward_calls=sum(calls.values()), forward_calls_by_phase=dict(calls))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work / "provenance"\n    spec = json.loads((provenance / "temporal_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    paths = backbone.history.acquire(args, spec["media_source"], spec["media_plan"])\n    cases, screening = screen_pairs(spec["cases"], spec["excluded"], paths, spec["expected_max_pairs"])\n    exports = work / "exports"\n    base.atomic(exports / "source_screen.json", dict(decisions=screening, source_status=spec["source_status"]))\n    if not cases:\n        raise RuntimeError("No pairs passed the shared-input and media-overlap screen")\n    print(f"SOURCE_SCREEN_PASS {len(cases)}/{spec[\'expected_max_pairs\']} pairs", flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, cuda=torch.version.cuda, torch=torch.__version__,\n                  gpu=torch.cuda.get_device_name(0), accepted=[c["pair_id"] for c in cases],\n                  window_span=WINDOW_SPAN, window_cap=WINDOW_CAP,\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    base.atomic(exports / "run_config.json", config)\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    checkpoints = work / "checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(cases):\n            pair = case["pair_id"]\n            path = checkpoints / f"{pair}.json"\n            if path.is_file():\n                saved = json.loads(path.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair\n                block = saved["block"]\n                print(f"PAIR_REUSED {i+1}/{len(cases)} {pair}", flush=True)\n            else:\n                torch.cuda.reset_peak_memory_stats()\n                branches = {b: branch_policy(model, paths[pair][b], case) for b in "ab"}\n                assert branches["a"]["sample"]["hashes"][-4:] == branches["b"]["sample"]["hashes"][-4:]\n                assert all(branches["a"]["answers"][o]["predictions"]["recent"] ==\n                           branches["b"]["answers"][o]["predictions"]["recent"] for o in base.ORDERS)\n                block = dict(pair_id=pair, branches=branches, peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n                base.atomic(path, dict(identity=identity, block=block))\n                print(f"PAIR_DONE {i+1}/{len(cases)} {pair}", flush=True)\n            blocks.append(block)\n    targets = json.loads((provenance / "evaluation_targets.json").read_bytes())\n    report = evaluate(blocks, targets)\n    base.atomic(exports / "summary.json", report)\n    base.atomic(work / "completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_TEMPORAL_WINDOWS_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work / "run.lock"), timeout=0):\n        (args.work / "exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work / "completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work / "completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n', 'provenance/refinement_cases.json': '{\n  "model_revision": "0d50680527681998e456c7b78950205bedd8a068",\n  "video_processor": {\n    "_processor_class": "LlavaOnevisionProcessor",\n    "do_pad": true,\n    "image_grid_pinpoints": [\n      [\n        384,\n        384\n      ],\n      [\n        384,\n        768\n      ],\n      [\n        384,\n        1152\n      ],\n      [\n        384,\n        1536\n      ],\n      [\n        384,\n        1920\n      ],\n      [\n        384,\n        2304\n      ],\n      [\n        768,\n        384\n      ],\n      [\n        768,\n        768\n      ],\n      [\n        768,\n        1152\n      ],\n      [\n        768,\n        1536\n      ],\n      [\n        768,\n        1920\n      ],\n      [\n        768,\n        2304\n      ],\n      [\n        1152,\n        384\n      ],\n      [\n        1152,\n        768\n      ],\n      [\n        1152,\n        1152\n      ],\n      [\n        1152,\n        1536\n      ],\n      [\n        1152,\n        1920\n      ],\n      [\n        1152,\n        2304\n      ],\n      [\n        1536,\n        384\n      ],\n      [\n        1536,\n        768\n      ],\n      [\n        1536,\n        1152\n      ],\n      [\n        1536,\n        1536\n      ],\n      [\n        1536,\n        1920\n      ],\n      [\n        1536,\n        2304\n      ],\n      [\n        1920,\n        384\n      ],\n      [\n        1920,\n        768\n      ],\n      [\n        1920,\n        1152\n      ],\n      [\n        1920,\n        1536\n      ],\n      [\n        1920,\n        1920\n      ],\n      [\n        1920,\n        2304\n      ],\n      [\n        2304,\n        384\n      ],\n      [\n        2304,\n        768\n      ],\n      [\n        2304,\n        1152\n      ],\n      [\n        2304,\n        1536\n      ],\n      [\n        2304,\n        1920\n      ],\n      [\n        2304,\n        2304\n      ]\n    ],\n    "image_processor_type": "LlavaOnevisionVideoProcessor",\n    "do_resize": true,\n    "size": {\n      "height": 384,\n      "width": 384\n    },\n    "resample": 3,\n    "do_rescale": true,\n    "rescale_factor": 0.00392156862745098,\n    "do_normalize": true,\n    "image_mean": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "image_std": [\n      0.5,\n      0.5,\n      0.5\n    ],\n    "do_convert_rgb": true\n  },\n  "media_source": {\n    "manifest": {\n      "dataset": "younght/TwinPast",\n      "revision": "c1f1cbab68bfda2982565c0491f1a97474b73558"\n    }\n  },\n  "media_plan": [\n    [\n      {\n        "pair_id": "twinpast_0208",\n        "video_a": "videos/twinpast_0208/A.mp4",\n        "video_b": "videos/twinpast_0208/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 19355906,\n            "sha256": "913c198fc475031b0431c84f05332c144925e30ae812ea1b7c4aabafcb81bd7b"\n          },\n          "b": {\n            "bytes": 19367971,\n            "sha256": "90da807127daed7dee910a83e831c8d468cc530754e945cc6c2011847812a102"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0094",\n        "video_a": "videos/twinpast_0094/A.mp4",\n        "video_b": "videos/twinpast_0094/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 23523559,\n            "sha256": "a4bc3db1a72cf2aa05deacedbcc79a25d60a9bb1e7181886600e08fc6dea4b89"\n          },\n          "b": {\n            "bytes": 22992602,\n            "sha256": "c78deb69dd5f496a65a527262fd48b9aa7e89821efd604893dcbd78984e1bc8c"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0099",\n        "video_a": "videos/twinpast_0099/A.mp4",\n        "video_b": "videos/twinpast_0099/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 27571576,\n            "sha256": "3612dacef08616a655a52b2cb80d729167adc6f6f1058ef0a76728a4bb07a167"\n          },\n          "b": {\n            "bytes": 29849686,\n            "sha256": "6ea5bb359efb0113e352a0e2ca3f0715a45fd088a3a411561606634e859f1639"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0252",\n        "video_a": "videos/twinpast_0252/A.mp4",\n        "video_b": "videos/twinpast_0252/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 38872585,\n            "sha256": "903f8e8ac3f39c4b504db4836d23631fa41e401642ada4fa115ee0fa4f2f1e73"\n          },\n          "b": {\n            "bytes": 39208455,\n            "sha256": "df5ec6d9f88971a2a6bf4a93bd5a8ff0be21b6aef15e8af1f7869abed3c692d5"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0095",\n        "video_a": "videos/twinpast_0095/A.mp4",\n        "video_b": "videos/twinpast_0095/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 13263369,\n            "sha256": "1b66c99709bd687ec4501e335e2e60487fb468b362210bfac182b3dabfde8a5e"\n          },\n          "b": {\n            "bytes": 13179586,\n            "sha256": "d18fad754703d8608aaad645da26d4a34ca2a67132bea66ef4b5533b2d37b1f5"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0259",\n        "video_a": "videos/twinpast_0259/A.mp4",\n        "video_b": "videos/twinpast_0259/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 20144782,\n            "sha256": "17b8f37fe3484646cfcfce19be92cbc32ad7c3a9a14fbde09be9e584587c1122"\n          },\n          "b": {\n            "bytes": 20325120,\n            "sha256": "b53459376e6138f03c8d3f57e8c55db0e8fcb6c8dcb05af49ae2bc3584573349"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0196",\n        "video_a": "videos/twinpast_0196/A.mp4",\n        "video_b": "videos/twinpast_0196/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 3713947,\n            "sha256": "62fc90176f7a14ffbf8524265d09b64cb4dc1f140c3ee065c19c9c484f18b706"\n          },\n          "b": {\n            "bytes": 3835416,\n            "sha256": "22192642f05331bae9da57228a0f6a611ebc3dddd3f0b76ccf628d8784f55027"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0185",\n        "video_a": "videos/twinpast_0185/A.mp4",\n        "video_b": "videos/twinpast_0185/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 4102201,\n            "sha256": "cad2075904aaad5577e1a9b60887708502e4a09f2829039420445eabd664dd2f"\n          },\n          "b": {\n            "bytes": 4122677,\n            "sha256": "f9b1001a8a8110ac6c49e7fda3e4c63f7ab1acdc5e07d0a3857f1bf7b526f8a2"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0214",\n        "video_a": "videos/twinpast_0214/A.mp4",\n        "video_b": "videos/twinpast_0214/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 25963688,\n            "sha256": "1f4ec4b343fd0ae0ab92cf07d61df32c71e8f56485d5d3ce0ed8095cb42985e1"\n          },\n          "b": {\n            "bytes": 25802733,\n            "sha256": "0682f3652c834a8868df6de722caae39ddac9fac6da6b9fcec08ed96b96b7c27"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0079",\n        "video_a": "videos/twinpast_0079/A.mp4",\n        "video_b": "videos/twinpast_0079/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 50061847,\n            "sha256": "9593bf6dd0ed8f11779767f627e5add09a935ef1a1854c4acda8649c389aad27"\n          },\n          "b": {\n            "bytes": 49883514,\n            "sha256": "7141fb5b3895869f0b7d258aea43dee43c55989c162b7e23aa6b7c2cd005acb9"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0236",\n        "video_a": "videos/twinpast_0236/A.mp4",\n        "video_b": "videos/twinpast_0236/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 16063112,\n            "sha256": "930820bc91fc7d02d8c3abb78c39897be89d5ea42c5cd4d1010a8a874b5d323d"\n          },\n          "b": {\n            "bytes": 15999399,\n            "sha256": "b9c9886edbced749b9bc6d6e326f564ab10bce9f5c34238c5441b99060f6f84a"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0036",\n        "video_a": "videos/twinpast_0036/A.mp4",\n        "video_b": "videos/twinpast_0036/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 17576106,\n            "sha256": "4fa6c986d8f9ccd3ca4446f1443e27a272f503cfaec5f48ffa83c2affc11efd9"\n          },\n          "b": {\n            "bytes": 17480443,\n            "sha256": "8ed2e5a5d7dec040cf207eadfa0523ab92498aee34e2762b468779f40fdbee92"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0001",\n        "video_a": "videos/twinpast_0001/A.mp4",\n        "video_b": "videos/twinpast_0001/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 13171396,\n            "sha256": "cfad5602a5b1167252d68fffca1b2a78a223455227f24c5b6a8be37efae95c9a"\n          },\n          "b": {\n            "bytes": 13217506,\n            "sha256": "97ad24e0d62bc8dab4d9b5acaa111eaa3a1d6a82ec6aa8a273b6882e2e90845b"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0037",\n        "video_a": "videos/twinpast_0037/A.mp4",\n        "video_b": "videos/twinpast_0037/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 10906636,\n            "sha256": "68371488689c87aed6a63f066ac8d75074b1010f1704f97cb00400481e40e646"\n          },\n          "b": {\n            "bytes": 10993223,\n            "sha256": "87049eeafddaac85b6b7d09c48e90cdcea07c827e3d050ac9de0f210be055ed0"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0153",\n        "video_a": "videos/twinpast_0153/A.mp4",\n        "video_b": "videos/twinpast_0153/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 8207871,\n            "sha256": "5b8b4ec98f8efa8138e7bf8934eb9c768a6b9c26bbc066423944d7a7a22c3fc9"\n          },\n          "b": {\n            "bytes": 8324053,\n            "sha256": "4cb31fb5e432908ffae28c1098d23f9bf9caf33784a9d12a2fe6751db9ef5299"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0031",\n        "video_a": "videos/twinpast_0031/A.mp4",\n        "video_b": "videos/twinpast_0031/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 9496513,\n            "sha256": "f611520139836f531c70765c06fc85a002d8f6e1d2f02b4f9656ac335558902c"\n          },\n          "b": {\n            "bytes": 9574043,\n            "sha256": "5299ccde4d090adc9bbccce56622559b6be2df1d8839e4bff32c2d07a1e007c0"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0291",\n        "video_a": "videos/twinpast_0291/A.mp4",\n        "video_b": "videos/twinpast_0291/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 11004021,\n            "sha256": "ba781165aeb06a4e9dc29b3988395b43e4c37a25cd1c46ed82bf908f2cb86056"\n          },\n          "b": {\n            "bytes": 10969101,\n            "sha256": "d4f3e0cd47b6ced30bd2c2cf06ab890176b97d7709339a8a23073cd63d9b0864"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0083",\n        "video_a": "videos/twinpast_0083/A.mp4",\n        "video_b": "videos/twinpast_0083/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 20102984,\n            "sha256": "66357c8c27082eb95a9ccadb32674b2a19f2a28f6c3fdb2c93154c9febfc49d9"\n          },\n          "b": {\n            "bytes": 20081207,\n            "sha256": "5143763f50177ef46d63e48a252ea5f5d066e64b6d676e54a0f747c6cd588d08"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0132",\n        "video_a": "videos/twinpast_0132/A.mp4",\n        "video_b": "videos/twinpast_0132/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 12169725,\n            "sha256": "e4027888640981f79e0df2c17e695962d0ff2513dea7d8125b7e7951756da260"\n          },\n          "b": {\n            "bytes": 12197287,\n            "sha256": "30af0773b972d76b69ea29204833798d6f8172c84433270490a6a6d340fd079f"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0326",\n        "video_a": "videos/twinpast_0326/A.mp4",\n        "video_b": "videos/twinpast_0326/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 38864193,\n            "sha256": "bc868588e810c64bc3f7ba42eee766a84e13c5d8d071e6b311df0df2be97ffa1"\n          },\n          "b": {\n            "bytes": 38793936,\n            "sha256": "db99c3048cae8d28243ca3295e924880cf54a3aea28fe992103208f27c812e28"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0255",\n        "video_a": "videos/twinpast_0255/A.mp4",\n        "video_b": "videos/twinpast_0255/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 16624513,\n            "sha256": "a5abe9605f5b8748ef6532db89ab6cbb55612f7db562d9609168e3cf135d64eb"\n          },\n          "b": {\n            "bytes": 16846600,\n            "sha256": "050cd5028cef8274781cddbaa38ed58b3de975613daf570795074febeb58c6da"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0118",\n        "video_a": "videos/twinpast_0118/A.mp4",\n        "video_b": "videos/twinpast_0118/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 15711648,\n            "sha256": "1e2c077d7960adadaecd54d5eacb5ddbfdb9fa4e0d760f90d44da17ec0671d6c"\n          },\n          "b": {\n            "bytes": 15785368,\n            "sha256": "e58b4b79362be9876edb284a9cf441a7f2aac8f1f5ad98619eab2a37c550f479"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0338",\n        "video_a": "videos/twinpast_0338/A.mp4",\n        "video_b": "videos/twinpast_0338/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 11553424,\n            "sha256": "f9ba9c04ee2b3ddaee8e05ea0a09cf596aea1943204cf70e41e8ccab68ef2b34"\n          },\n          "b": {\n            "bytes": 11528085,\n            "sha256": "33e0c317de9f05651a3640adb419a5ad76a694056022446609db18d0c09c7e82"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0164",\n        "video_a": "videos/twinpast_0164/A.mp4",\n        "video_b": "videos/twinpast_0164/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 7044647,\n            "sha256": "2aae7b2a1d8717fa50800c33f3345b69f12f100121ba7b742d122514d24e0a82"\n          },\n          "b": {\n            "bytes": 7090016,\n            "sha256": "80db2aca755771b394ba1d72f875663144c80e637d7ccfd8fb35b8605fc87d3d"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0120",\n        "video_a": "videos/twinpast_0120/A.mp4",\n        "video_b": "videos/twinpast_0120/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 25283725,\n            "sha256": "9bcbd0ff510845f5385ecb9aecd0e63f160fb3f70619dbc57dc7a9acb8fd70b3"\n          },\n          "b": {\n            "bytes": 26609694,\n            "sha256": "1f701a8e5746e67c3aa6abded420da371339d069998cd8e743f856a96f8cc199"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0085",\n        "video_a": "videos/twinpast_0085/A.mp4",\n        "video_b": "videos/twinpast_0085/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 4608254,\n            "sha256": "23909dece7fad9af70b53e8b1b333057600095ceaf999d1d2d4c485b077348ca"\n          },\n          "b": {\n            "bytes": 4651289,\n            "sha256": "06ea2b4e280cabc773f7b2c0566f8f7890f52d8bc293e74a85d757aa6b256494"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0206",\n        "video_a": "videos/twinpast_0206/A.mp4",\n        "video_b": "videos/twinpast_0206/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 7534621,\n            "sha256": "dec07694bf90c232ef4b26ddf824e42fd805c5e3b6ffe99821fe4c7d3bacefbb"\n          },\n          "b": {\n            "bytes": 7541950,\n            "sha256": "e0799ab8db777a0d67a0ea176177b8937bfd750c318fafaa6a0793a4a923a1e1"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0021",\n        "video_a": "videos/twinpast_0021/A.mp4",\n        "video_b": "videos/twinpast_0021/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 24797005,\n            "sha256": "96f47091116bbad5d8bc4c2dda471255372950e5565ff063d57f09e735a812f6"\n          },\n          "b": {\n            "bytes": 24575581,\n            "sha256": "6d2a1d5f22652301859dffbcadba8e970d597538ddc75007fba1ff413b63fbf3"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0272",\n        "video_a": "videos/twinpast_0272/A.mp4",\n        "video_b": "videos/twinpast_0272/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 17241368,\n            "sha256": "c3347c2951f17cdde6343209455699818846c9c3f37a01b0d1a4175559d29e3b"\n          },\n          "b": {\n            "bytes": 17053208,\n            "sha256": "538cda9cefa318cfce3da67c49244b69abe77d19d135445e11c441b3314eb260"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0204",\n        "video_a": "videos/twinpast_0204/A.mp4",\n        "video_b": "videos/twinpast_0204/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 6370525,\n            "sha256": "66376858cc6fce8780015a53117b45ae6bad5365bf0f546acac1d6ea80cbb56a"\n          },\n          "b": {\n            "bytes": 6356519,\n            "sha256": "7692ff8b1ab5db456c1f4f282c8898fc82382a287cd6a2a48919ebfe25d17ce3"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0141",\n        "video_a": "videos/twinpast_0141/A.mp4",\n        "video_b": "videos/twinpast_0141/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 11005563,\n            "sha256": "5f01a224b8eab263f3482f9e57516b8ff379e90b476013d52af566cba6734378"\n          },\n          "b": {\n            "bytes": 11051711,\n            "sha256": "aae370b7a451402585a091247344da41dcfabaccd658f578cc631424e97613a6"\n          }\n        }\n      },\n      null\n    ],\n    [\n      {\n        "pair_id": "twinpast_0107",\n        "video_a": "videos/twinpast_0107/A.mp4",\n        "video_b": "videos/twinpast_0107/B.mp4"\n      },\n      {\n        "decoded": {\n          "a": {\n            "bytes": 15912952,\n            "sha256": "c0f429f59bec0868cf5e80e377b460c361b2fcdc2290c5e556e83e3417955586"\n          },\n          "b": {\n            "bytes": 15898401,\n            "sha256": "8e00de3bf38fe4f87c922638057fd4cfe491beb17edce7982e75ec38026297e9"\n          }\n        }\n      },\n      null\n    ]\n  ],\n  "reference_ids": [\n    "twinpast_0208",\n    "twinpast_0094",\n    "twinpast_0099",\n    "twinpast_0252",\n    "twinpast_0095",\n    "twinpast_0259",\n    "twinpast_0196",\n    "twinpast_0185",\n    "twinpast_0214",\n    "twinpast_0079",\n    "twinpast_0236",\n    "twinpast_0036",\n    "twinpast_0001",\n    "twinpast_0037",\n    "twinpast_0153",\n    "twinpast_0031",\n    "twinpast_0291",\n    "twinpast_0083",\n    "twinpast_0132",\n    "twinpast_0326"\n  ],\n  "excluded_ids": [\n    "twinpast_0208",\n    "twinpast_0094",\n    "twinpast_0099",\n    "twinpast_0252",\n    "twinpast_0095",\n    "twinpast_0259",\n    "twinpast_0196",\n    "twinpast_0185",\n    "twinpast_0214",\n    "twinpast_0079",\n    "twinpast_0236",\n    "twinpast_0036",\n    "twinpast_0001",\n    "twinpast_0037",\n    "twinpast_0153",\n    "twinpast_0031",\n    "twinpast_0291",\n    "twinpast_0083",\n    "twinpast_0132",\n    "twinpast_0326",\n    "twinpast_0166",\n    "twinpast_0162",\n    "twinpast_0249",\n    "twinpast_0034"\n  ],\n  "source_status": "Original source IDs unavailable. Sampled media screening is a duplicate proxy.",\n  "expected_max_pairs": 8,\n  "cases": [\n    {\n      "pair_id": "twinpast_0255",\n      "query_s": 35.5,\n      "question": "In the earlier clip, what was the order of the letters at the beginning?",\n      "options": [\n        "sel",\n        "home"\n      ]\n    },\n    {\n      "pair_id": "twinpast_0118",\n      "query_s": 45.625,\n      "question": "At the end of the guitar-case scene, before the player in a green shirt appears, was the case cover lowered over the guitar body or lifted away from it?",\n      "options": [\n        "Lowered over the guitar body.",\n        "Lifted away from the guitar body."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0338",\n      "query_s": 39.208333333333336,\n      "question": "At the end of the earlier tiled-floor segment, immediately before the scene changes, was the small yellow box visible or hidden beneath the red cover?",\n      "options": [\n        "Hidden beneath the red cover.",\n        "Visible on the floor."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0164",\n      "query_s": 22.041666666666668,\n      "question": "At the end of the close-up of the orange publication on the stone counter, was the publication open or closed?",\n      "options": [\n        "The publication was open.",\n        "The publication was closed."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0120",\n      "query_s": 42.958333333333336,\n      "question": "At the end of the box-packing scene, before the video moves on to packing a backpack, what is the visible state of the box top?",\n      "options": [\n        "The top flaps lay flat across the opening, covering the contents.",\n        "The top flaps remained raised and the contents were exposed."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0085",\n      "query_s": 39.041666666666664,\n      "question": "At the end of the earlier bedding segment, immediately before the scene changes, was the white pillow exposed or covered by the gray sheet?",\n      "options": [\n        "Covered by the gray sheet.",\n        "Exposed beside the gray sheet."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0206",\n      "query_s": 20.958333333333332,\n      "question": "In the wooden-table scene with the green bottle, what was the large round blue lid resting on just before the scene changed?",\n      "options": [\n        "The tall translucent pitcher.",\n        "The small rectangular clear container."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0021",\n      "query_s": 66.95833333333333,\n      "question": "In the last view of the orange bowl resting on the counter before the eggs are poured into the frying pan, are separate yolks still visible, or have the eggs been beaten into a blended mixture?",\n      "options": [\n        "Separate yolks were still visible.",\n        "The eggs had been beaten into a blended mixture."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0272",\n      "query_s": 38.958333333333336,\n      "question": "At the end of the opening tabletop scene with a black travel mug and two blue bowls, which bowl was underneath the larger translucent lid?",\n      "options": [\n        "The larger bowl.",\n        "The smaller bowl."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0204",\n      "query_s": 22.791666666666668,\n      "question": "At the end of the scene where the person holds a blue bottle in front of the camera, was the cap on or off the bottle?",\n      "options": [\n        "The cap was on the bottle.",\n        "The cap was off the bottle."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0141",\n      "query_s": 29.125,\n      "question": "Where was the pen at the end of the mug scene, before the pen-sorting scene resumed?",\n      "options": [\n        "To the left of the mug.",\n        "Behind the mug, with its top still visible."\n      ]\n    },\n    {\n      "pair_id": "twinpast_0107",\n      "query_s": 38.958333333333336,\n      "question": "At the end of the opening wooden-table scene with a tall glass soda bottle, what was the large black domed lid resting on?",\n      "options": [\n        "The white cylindrical flask.",\n        "The clear plastic bottle."\n      ]\n    }\n  ],\n  "manifest_sha256": "3ef783e6f7df0a78ca945956d3f7c127f5cc457125df0ba43c0d5ed61d1e6370",\n  "subsecond_archive_sha256": "5c4ae82d43bea514074bd9bf375912a39362203c308b387bb8c8e0f02844f10a"\n}\n', 'provenance/evaluation_targets.json': '{\n  "twinpast_0021": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0085": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0107": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0118": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0120": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0141": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0164": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0204": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0206": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0255": {\n    "answer_a": "A",\n    "answer_b": "B"\n  },\n  "twinpast_0272": {\n    "answer_a": "B",\n    "answer_b": "A"\n  },\n  "twinpast_0338": {\n    "answer_a": "A",\n    "answer_b": "B"\n  }\n}\n', 'provenance/refinement_runner.py': '"""Compare change-directed and uniform refinement in the same causal windows."""\nimport argparse\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nimport temporal_runner as temporal\n\nbase = temporal.base\nbackbone = temporal.backbone\nARTIFACT = "HERMES_twinpast_refinement_evidence.zip"\nMETHODS = ("coarse_uniform", "local_uniform", "state_change")\n\n\ndef acquire(args, spec):\n    paths, pending = {}, []\n    with ThreadPoolExecutor(max_workers=3) as pool:\n        for plan, media, _ in spec["media_plan"]:\n            pair = plan["pair_id"]\n            paths[pair] = {}\n            for branch, receipt in media["decoded"].items():\n                name = Path(plan[f"video_{branch}"])\n                options = [args.previous / "media" / name,\n                           args.work.with_name("hermes_twinpast_diagnostic") / "media" / name,\n                           args.media_reuse / name.relative_to("videos")]\n                match = next((p for p in options if p.is_file() and base.digest(p) == receipt["sha256"]), None)\n                if match is not None:\n                    paths[pair][branch] = match\n                    print(f"MEDIA_REUSED {name.as_posix()}", flush=True)\n                else:\n                    dest = paths[pair][branch] = args.work / "media" / name\n                    pending.append(pool.submit(base.fetch, spec["media_source"], name.as_posix(), dest,\n                                               receipt["sha256"], receipt["bytes"]))\n        for download in pending:\n            download.result()\n    return paths\n\n\ndef refine(path, windows):\n    import numpy as np\n    from PIL import Image\n\n    grid = [start + i/4 for start, end in windows for i in range(4*(end-start)+1)]\n    frames, trace = temporal.decode_grid(path, grid)\n    rows, picks = [], {"local_uniform": [], "state_change": []}\n    for k in range(3):\n        origin = 13*k\n        chunk = frames[origin:origin+13]\n        rgb = np.stack([np.asarray(Image.fromarray(f).resize((32, 18)), dtype=np.float32) for f in chunk])\n        changes = np.abs(np.diff(rgb, axis=0)).mean(axis=(1,2,3)).tolist()\n        peak = max(range(len(changes)), key=changes.__getitem__)\n        uniform = [origin+3, origin+9]\n        changed = [origin+peak, origin+peak+1]\n        picks["local_uniform"].extend(uniform)\n        picks["state_change"].extend(changed)\n        rows.append(dict(window_s=windows[k], adjacent_rgb_mae=changes, uniform_slots=uniform,\n                         change_slots=changed))\n    assert all(len(v) == len(set(v)) == 6 for v in picks.values())\n    return frames, trace, rows, picks\n\n\ndef branch_policy(model, path, case):\n    import torch\n\n    tick = time.perf_counter()\n    frames, coarse = temporal.decode_grid(path, range(math.ceil(case["query_s"])))\n    n = len(frames)-4\n    bank = temporal.auto.encode_frames(model, frames)\n    recent = bank[-4:]\n    system = model.language_model.get_input_embeddings()(torch.as_tensor(model.init_prompt_ids, device=model.device))\n    calls = []\n\n    def score(prompt, visual, arm, order):\n        ids, option_ids, perm, sha = prompt\n        start = time.perf_counter()\n        values, tokens = base.score(model, system, ids, option_ids, visual)\n        canonical = [0.0]*len(perm)\n        for i,v in zip(perm, values.tolist(), strict=True):\n            canonical[i] = v\n        calls.append(dict(phase=arm, order=order, forward_s=time.perf_counter()-start, lm_tokens=tokens,\n                          visual_tokens=visual.shape[1], prompt_sha256=sha))\n        return canonical\n\n    probe = ("Does this earlier excerpt contain visual evidence needed to answer the following "\n             "question about the specified event, state, or entity? " + case["question"])\n    probes = {o: base.make_prompt(model, probe, ["Yes", "No"], o, True) for o in base.ORDERS}\n    candidates = []\n    for start in temporal.auto.spread(n-temporal.WINDOW_SPAN, temporal.WINDOW_CAP):\n        stop = start+temporal.WINDOW_SPAN\n        visual = torch.cat([bank[start], bank[stop]], dim=1)\n        raw = {o: score(probes[o], visual, "shared_retrieval", o) for o in base.ORDERS}\n        relevance = sum(v[0]-v[1] for v in raw.values()) / 2\n        candidates.append(dict(slots=[start, stop], relevance=relevance, yes_no=raw))\n    selected = temporal.select_windows(candidates)\n    windows = [selected[i:i+2] for i in range(0, 6, 2)]\n    samples, fine, changes, picks = refine(path, windows)\n    assert len(samples)==39 and fine["timestamps_s"][-1]<coarse["timestamps_s"][-4]<case["query_s"]\n    coarse_slots = temporal.auto.spread(n,6)\n    features = {"coarse_uniform": [bank[i] for i in coarse_slots]}\n    for arm, use in picks.items():\n        features[arm] = temporal.auto.encode_frames(model, samples[use])\n    inputs = {arm: torch.cat(v+recent, dim=1) for arm,v in features.items()}\n    assert len({tuple(x.shape) for x in inputs.values()})==1\n    answers = {}\n    for o in base.ORDERS:\n        prompt = base.make_prompt(model, case["question"], case["options"], o, True)\n        logits = {m: score(prompt, inputs[m], m, o) for m in METHODS}\n        answers[o] = {m: dict(logits=v, prediction=base.LETTERS[temporal.auto.best(v)]) for m,v in logits.items()}\n    return dict(coarse=coarse, candidates=candidates, selected_windows=windows, refined=fine, changes=changes,\n                coarse_uniform_slots=coarse_slots, refined_picks=picks, answers=answers, calls=calls,\n                inspected_refinement_frames=len(samples), selected_frames_per_arm=6, wall_s=time.perf_counter()-tick)\n\n\ndef summarize(blocks, targets):\n    methods = {}\n    for arm in METHODS:\n        robust, correct = [], 0\n        for b in blocks:\n            pair = b["pair_id"]\n            answers = [b["branches"][side]["answers"][o][arm]["prediction"]==targets[pair][f"answer_{side}"]\n                       for side in "ab" for o in base.ORDERS]\n            correct += sum(answers)\n            if all(answers):\n                robust.append(pair)\n        methods[arm] = dict(robust_pairs=robust, correct_branch_orders=correct)\n    change = set(methods["state_change"]["robust_pairs"])\n    uniform = set(methods["local_uniform"]["robust_pairs"])\n    calls = Counter(c["phase"] for r in blocks for b in r["branches"].values() for c in b["calls"])\n    return dict(pairs=len(blocks), methods=methods, state_change_only=sorted(change-uniform),\n                local_uniform_only=sorted(uniform-change), state_change_net_robust_gain=len(change)-len(uniform),\n                forward_calls_by_phase=dict(calls))\n\n\ndef execute(args):\n    import torch\n    from inference.llavaov_hermes import load_model\n\n    work = args.work\n    provenance = work/"provenance"\n    spec = json.loads((provenance/"refinement_cases.json").read_bytes())\n    assert args.model.name == backbone.MODEL_REVISION == spec["model_revision"]\n    paths = acquire(args, spec)\n    cases, decisions = temporal.screen_pairs(spec["cases"], spec["reference_ids"], paths, spec["expected_max_pairs"])\n    exports = work/"exports"\n    base.atomic(exports/"source_screen.json", dict(decisions=decisions, source_status=spec["source_status"]))\n    if not cases:\n        raise RuntimeError("No pairs passed the media and recent-frame screen")\n    print(f"SOURCE_SCREEN_PASS {len(cases)}/{spec[\'expected_max_pairs\']} pairs", flush=True)\n    model, processor = load_model(model_path=str(args.model), kv_size=4000, streaming=True, sample_fps=0.5)\n    assert model.dtype == torch.float16 and {str(p.device) for p in model.parameters()} == {"cuda:0"}\n    assert json.loads(json.dumps(processor.video_processor.to_dict())) == spec["video_processor"]\n    config = dict(model_revision=args.model.name, cuda=torch.version.cuda, torch=torch.__version__,\n                  gpu=torch.cuda.get_device_name(0), accepted=[c["pair_id"] for c in cases],\n                  hashes={p.name: base.digest(p) for p in sorted(provenance.iterdir()) if p.is_file()})\n    identity = base.sha(json.dumps(config, sort_keys=True).encode())\n    base.atomic(exports/"run_config.json", config)\n    checkpoints = work/"checkpoints"\n    checkpoints.mkdir(exist_ok=True)\n    blocks = []\n    with torch.inference_mode():\n        for i, case in enumerate(cases):\n            pair = case["pair_id"]\n            cp = checkpoints/f"{pair}.json"\n            if cp.is_file():\n                saved = json.loads(cp.read_bytes())\n                assert saved["identity"] == identity and saved["block"]["pair_id"] == pair\n                block = saved["block"]\n            else:\n                torch.cuda.reset_peak_memory_stats()\n                branches = {b: branch_policy(model, paths[pair][b], case) for b in "ab"}\n                assert branches["a"]["coarse"]["hashes"][-4:] == branches["b"]["coarse"]["hashes"][-4:]\n                block = dict(pair_id=pair, branches=branches, peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated())\n                base.atomic(cp, dict(identity=identity, block=block))\n            blocks.append(block)\n            print(f"PAIR_DONE {i+1}/{len(cases)} {pair}", flush=True)\n    targets = json.loads((provenance/"evaluation_targets.json").read_bytes())\n    report = summarize(blocks, targets)\n    base.atomic(exports/"summary.json", report)\n    base.atomic(work/"completion.json", dict(status="complete", pairs=len(blocks)))\n    print("TWINPAST_REFINEMENT_COMPLETE", json.dumps(report), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    for name in ("work", "model", "media-reuse", "previous"):\n        parser.add_argument(f"--{name}", type=Path, required=True)\n    args = parser.parse_args()\n    from filelock import FileLock\n\n    with FileLock(str(args.work/"run.lock"), timeout=0):\n        (args.work/"exports").mkdir(exist_ok=True)\n        base.ARTIFACT = ARTIFACT\n        base.atomic(args.work/"completion.json", dict(status="running"))\n        try:\n            execute(args)\n        except BaseException as exc:\n            base.atomic(args.work/"completion.json", dict(status="failed", error_type=type(exc).__name__, message=str(exc)))\n            raise\n        finally:\n            base.package(args.work)\n'}
EXPECTED = {'pinned_source/inference/llavaov_hermes.py': '45fee483fbc37fe65433c84146eba42fc18dae40f6878c7a2f50fd8e55a387f0', 'pinned_source/inference/abstract_hermes.py': 'cbf62ad9f6dada32db598d7c37c27f995bbb37375e12a282a88cf4426f495dc0', 'pinned_source/inference/reindex_1d.py': '22ca0dee63cd7a4d00161f1fc0485143a57145862b081555598b904bbc95e0b0', 'provenance/twinpast_base.py': '5f4e15c54168502763fb8ad2054e6c701e7440a09295b01240ef05e3865066f0', 'provenance/history_only_runner.py': '7614a0f8f72dd1e3e92e7b0a9a7e8ed232a1ab6778fac9a910fd48171c70f281', 'provenance/backbone_runner.py': '6a6f51a49f1ac1ce2f978f615b6a51a2b26404ca60147cf0b74f2c3b8f187ce4', 'provenance/automatic_runner.py': '6a2f33bb3907aea3a0b2b5aa9bdcd4bd6cb20c000f77247b3d11e1907378ecd3', 'provenance/temporal_runner.py': '5d2532993e68d2b378c395e75fb099ec7afaf32f54ca380a0c52541e84d01ee1', 'provenance/refinement_cases.json': '1d11234a60a495d4ac726094a2539cfdffec6db44462fe3e50ed9a527c790c95', 'provenance/evaluation_targets.json': 'e0d2866a001b8a5ee6832d4348d90447b32a8eccd4a4a6781f7278cedb0152a6', 'provenance/refinement_runner.py': '6c47e4d8989c4bea9a3e8f8de6cd073831582ba9887f8426bfc120a32cd9e731'}

for relative, contents in SOURCES.items():
    raw = contents.encode('utf-8')
    assert hashlib.sha256(raw).hexdigest() == EXPECTED[relative]
    if relative.endswith('.py'):
        ast.parse(contents, filename=relative)
    if not VALIDATE_ONLY:
        target = WORK / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and target.read_bytes() != raw:
            raise ValueError(f'Existing file differs at {target}. Preserve it and choose a fresh WORK directory.')
        if not target.is_file():
            target.write_bytes(raw)
print('EMBEDDED_SOURCE_AND_MANIFEST_PASS', BASE_REVISION)


EMBEDDED_SOURCE_AND_MANIFEST_PASS 2a1385e0abc94debb64761b0465713c799a81c1c


In [ ]:
ENV_PROBE = r"""
import sys, torch, transformers, tokenizers, huggingface_hub, pandas, decord, logzero
import json, importlib.metadata as md
assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == '2.10.0+cu128'
assert transformers.__version__ == '4.45.0.dev0'
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert '66bc4def9505fa7c7fe4aa7a248c34a026bb552b' in str(origin), 'Transformers source revision differs'
assert tokenizers.__version__ == '0.19.1' and huggingface_hub.__version__ == '0.34.4'
assert torch.cuda.is_available(), 'PyTorch cannot use CUDA; check the container driver'
from transformers.models.qwen2.modeling_qwen2 import Qwen2RotaryEmbedding
import inspect
assert 'seq_len' in inspect.signature(Qwen2RotaryEmbedding.forward).parameters
x = torch.ones((16, 16), device='cuda')
assert float((x @ x)[0, 0]) == 16
print('PINNED_ENV_PASS', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
"""

if not VALIDATE_ONLY:
    candidate_python = REUSE / 'venv/bin/python'
    if candidate_python.is_file() and probe([candidate_python, '-c', ENV_PROBE], ENV).returncode == 0:
        PYTHON = candidate_python
        print('REUSING_VERIFIED_PINNED_ENVIRONMENT', PYTHON, flush=True)

    if not PYTHON.is_file() or probe([PYTHON, '-c', ENV_PROBE], ENV).returncode:
        downloads = WORK / 'downloads'
        local_wheels = [
            downloads / 'torch-2.10.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl',
            downloads / 'nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl',
            downloads / 'nvidia_nccl_cu12-2.27.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
            downloads / 'nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
            downloads / 'nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl',
        ]
        missing_wheels = [str(path) for path in local_wheels if not path.is_file()]
        if missing_wheels:
            raise FileNotFoundError('Missing local wheel(s):\n' + '\n'.join(missing_wheels))

        PIP_ENV = package_environment(ENV)
        if probe([sys.executable, '-m', 'uv', '--version'], ENV).returncode:
            attempt = probe(
                [sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                 'https://pypi.tuna.tsinghua.edu.cn/simple'],
                PIP_ENV,
            )
            if attempt.returncode:
                logged(
                    [sys.executable, '-m', 'pip', 'install', '-q', 'uv', '-i',
                     'https://mirrors.aliyun.com/pypi/simple'],
                    'setup.log', PIP_ENV,
                )

        UV = [sys.executable, '-m', 'uv', '--verbose']
        if not PYTHON.is_file():
            logged(UV + ['venv', VENV, '--python', '3.12', '--seed'], 'setup.log', ENV)

        def install(packages, extra=()):
            if extra:
                install_packages(UV, PYTHON, packages, ENV, extra)
            else:
                logged(
                    UV + [
                        'pip', 'install', '--python', PYTHON,
                        '--no-index', '--find-links', str(downloads),
                    ] + list(packages),
                    'setup.log', ENV,
                )

        for wheel in local_wheels:
            print('USING_LOCAL_WHEEL', wheel.name, flush=True)
        install([str(wheel) for wheel in local_wheels])

        install([
            'numpy==1.26.4', 'tokenizers==0.19.1', 'huggingface-hub==0.34.4',
            'accelerate==0.34.2', 'decord==0.6.0', 'logzero', 'pillow',
            'sentencepiece', 'safetensors', 'packaging', 'filelock', 'regex',
            'requests', 'pyyaml', 'tqdm', 'fsspec', 'protobuf', 'einops',
            'psutil', 'pandas',
        ])

        tf_archive = downloads / f'transformers-{TF_SHA}.zip'
        if not tf_archive.is_file() or not zipfile.is_zipfile(tf_archive):
            fetch_tf = r"""
import sys, urllib.request, zipfile, os, time
from pathlib import Path

dest = Path(sys.argv[2])
temp = dest.with_suffix('.download')
for attempt in range(4):
    try:
        with urllib.request.urlopen(sys.argv[1], timeout=180) as source, temp.open('wb') as target:
            while True:
                block = source.read(1024 * 1024)
                if not block:
                    break
                target.write(block)
        assert zipfile.is_zipfile(temp), 'Transformers source is not a ZIP'
        os.replace(temp, dest)
        break
    except Exception:
        if attempt == 3:
            raise
        time.sleep(3 * (attempt + 1))
"""
            logged(
                [sys.executable, '-c', fetch_tf,
                 f'https://codeload.github.com/huggingface/transformers/zip/{TF_SHA}',
                 tf_archive],
                'source_download.log', ENV,
            )
        install([str(tf_archive)], ['--no-deps'])

    logged([PYTHON, '-c', ENV_PROBE], 'environment.log', ENV)
    ENV['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO / 'scripts')
    logged([PYTHON, '-m', 'pip', 'freeze'], 'environment_freeze.txt', ENV)

USING_LOCAL_WHEEL torch-2.10.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl
USING_LOCAL_WHEEL nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl
USING_LOCAL_WHEEL nvidia_nccl_cu12-2.27.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
USING_LOCAL_WHEEL nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
USING_LOCAL_WHEEL nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl
SETUP_STAGE setup.log
DEBUG Searching for user configuration in: /root/.config/uv/uv.toml
DEBUG uv 0.13.10 (x86_64-unknown-linux-gnu)
DEBUG Checking for Python interpreter at path `/root/autodl-tmp/hermes_twinpast_refinement/venv/bin/python`
Using Python 3.12.3 environment at: /root/autodl-tmp/hermes_twinpast_refinement/venv
DEBUG At least one requirement is not satisfied: file:///root/autodl-tmp/hermes_twinpast_refinement/downloads/torch-2.10.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl
DEBUG Using request connect timeout of 10s and read timeout of 30s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



DEBUG adding 'transformers/models/mixtral/convert_mixtral_weights_to_hf.py'
DEBUG adding 'transformers/models/mixtral/modeling_mixtral.py'
DEBUG adding 'transformers/models/mluke/__init__.py'
DEBUG adding 'transformers/models/mluke/convert_mluke_original_pytorch_checkpoint_to_pytorch.py'
DEBUG adding 'transformers/models/mluke/tokenization_mluke.py'
DEBUG adding 'transformers/models/mobilebert/__init__.py'
DEBUG adding 'transformers/models/mobilebert/configuration_mobilebert.py'
DEBUG adding 'transformers/models/mobilebert/convert_mobilebert_original_tf_checkpoint_to_pytorch.py'
DEBUG adding 'transformers/models/mobilebert/modeling_mobilebert.py'
DEBUG adding 'transformers/models/mobilebert/modeling_tf_mobilebert.py'
DEBUG adding 'transformers/models/mobilebert/tokenization_mobilebert.py'
DEBUG adding 'transformers/models/mobilebert/tokenization_mobilebert_fast.py'
DEBUG adding 'transformers/models/mobilenet_v1/__init__.py'
DEBUG adding 'transformers/models/mobilenet_v1/configuration_m

In [ ]:
# Standalone notebook cell. Uses WORK, ENV, MODEL_ID, MODEL_SHA and logged from setup.
import hashlib
import json
import re
import shutil
from pathlib import Path

SHARDS = (
    ("model-00001-of-00004.safetensors", 4911200360, "ad4cb27d2a3f6e987147619cf2508a5bbb1b6b4c2d5533501fcc4a287a90e5af"),
    ("model-00002-of-00004.safetensors", 4991497664, "cfaed9e4ba187283a936fd6df5d55148a6de5a8416f1279778b364232c05f39a"),
    ("model-00003-of-00004.safetensors", 4932752752, "86cb0bb7f7d57873b27f16fd66b54be71055e582a48ce26e30884bfb209dffd4"),
    ("model-00004-of-00004.safetensors", 1226266240, "a2d1e40e47e9b524efcbbcab68e97fe4e7aadfe9a51bc14b54cc2331c48ebff6"),
)
METADATA = (
    ("added_tokens.json", 122, "33e0fb93dadacb864bd2f2e8441e147daa2baceb67f94d3ef5283b495572cea0"),
    ("chat_template.json", 826, "2466d1704df30f0067f28d8e30e0190a1bf74e5b430942697af974d162a056bd"),
    ("config.json", 2528, "69277c2c9ba8a4f61d2e72f79fbe02e043b8c3af670858a606818527f971a0c2"),
    ("generation_config.json", 126, "89dc53229f50b59570b6852056dafeac8116c458f1a748bff491b6d4d24d3b51"),
    ("merges.txt", 1671853, "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"),
    ("model.safetensors.index.json", 77952, "e389b969d3b9b7120f136fbf6592cbbc1c07326157cae6f863fb64559261dae9"),
    ("preprocessor_config.json", 1732, "3644c108b9f0fa53e62ff422a9be6639642f0e64dab4a71f961c7911d4386384"),
    ("processor_config.json", 178, "04e9899e93f2a412c94e153cab4081457c9a44defb3b2c0b9df673d42c42cdd0"),
    ("special_tokens_map.json", 367, "f4f79e08d97f4d1c87f8d89264f525c8789da3b73b3bb55d1e12f692f41a7b1b"),
    ("tokenizer.json", 7028579, "3c0ce3213b50ff38d8aa1e91136a2d2cb142a3f569246170872e439cb2a29d15"),
    ("tokenizer_config.json", 1801, "b3151cdc153a44fa4e28abc8fc83567fc663aef7cd9dc041f5d66e617fe10b8e"),
    ("video_preprocessor_config.json", 621, "0ea9b672282b78353960b6069a521ed496a9f07c033e1e3362bff669234caa8d"),
    ("video_processor/preprocessor_config.json", 428, "1e71b2e75b90ddf696692529485b4a75fd54ecd4bbc03e2cc7be4af032875765"),
    ("vocab.json", 2776833, "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"),
)


def file_bytes(path):
    return path.stat().st_size if path.is_file() else 0


def verified(path, size, expected):
    if file_bytes(path) != size:
        return False
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest() == expected


def range_matches(headers, offset, end, total):
    replies = re.split(r"(?m)(?=^HTTP/)", headers.strip())
    final = replies[-1]
    return (re.match(r"HTTP/\S+\s+206\b", final) is not None
            and re.search(rf"(?im)^content-range:\s*bytes {offset}-{end}/{total}\s*$", final) is not None)


def fetch_pinned(name, size, expected, destination):
    if verified(destination, size, expected):
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".incomplete")
    direct_env = {k: v for k, v in ENV.items() if k.lower() not in {
        "http_proxy", "https_proxy", "all_proxy", "no_proxy"}}
    endpoints = (
        (f"https://modelscope.cn/models/{MODEL_ID}/resolve/master/{name}", direct_env),
        (f"https://hf-mirror.com/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
        (f"https://huggingface.co/{MODEL_ID}/resolve/{MODEL_SHA}/{name}", ENV),
    )
    offset = file_bytes(partial)
    if offset > size:
        raise RuntimeError(f"Oversized partial file: {partial} ({offset} bytes, expected {size})")
    chunk = partial.with_suffix(".range")
    headers = partial.with_suffix(".headers")
    while offset < size:
        end = min(offset + 64 * 1024**2, size) - 1
        length = end - offset + 1
        for url, env in endpoints:
            host = url.split("/")[2]
            print(f"MODEL_RANGE {name} bytes={offset}-{end}/{size} source={host}", flush=True)
            try:
                logged(["curl", "--fail", "--location", "--range", f"{offset}-{end}", "--connect-timeout", "15",
                        "--speed-limit", "16384", "--speed-time", "90", "--max-time", "600",
                        "--max-filesize", str(length), "--dump-header", headers, "--output", chunk,
                        "--progress-bar", url], "model_download.log", env)
            except RuntimeError:
                print(f"MODEL_ENDPOINT_RETRY {name} source={host}", flush=True)
                continue
            response = headers.read_text(encoding="iso-8859-1")
            # Some mirrors answer a request for an entire small file with a plain HTTP 200.
            whole_file = offset == 0 and length == size and verified(chunk, size, expected)
            if file_bytes(chunk) != length or not (whole_file or range_matches(response, offset, end, size)):
                print(f"MODEL_RANGE_REJECTED {name} source={host}", flush=True)
                continue
            with partial.open("ab") as output, chunk.open("rb") as incoming:
                if output.tell() != offset:
                    raise RuntimeError(f"Another downloader changed {partial}. Stop concurrent downloads.")
                try:
                    shutil.copyfileobj(incoming, output, length=1024**2)
                except BaseException:
                    output.flush()
                    output.truncate(offset)
                    raise
            offset += length
            print(f"MODEL_PROGRESS {name} {offset}/{size} bytes", flush=True)
            break
        else:
            raise RuntimeError(f"All endpoints failed for {name} at byte {offset}. Rerun to resume {partial}.")
    chunk.unlink(missing_ok=True)
    headers.unlink(missing_ok=True)
    if not verified(partial, size, expected):
        raise RuntimeError(f"SHA-256 mismatch in {partial}. The file was kept for inspection.")
    partial.replace(destination)


if not VALIDATE_ONLY:
    assert MODEL_ID == "llava-hf/llava-onevision-qwen2-7b-ov-hf"
    assert MODEL_SHA == "0d50680527681998e456c7b78950205bedd8a068"
    repo = Path(ENV["HF_HOME"]) / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    blobs = repo / "blobs"
    MODEL_PATH = repo / "snapshots" / MODEL_SHA
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    missing = sum(max(0, size - file_bytes(blobs / (sha + ".incomplete")))
                  for name, size, sha in SHARDS
                  if file_bytes(MODEL_PATH / name) != size and file_bytes(blobs / sha) != size)
    free = shutil.disk_usage(MODEL_PATH).free
    assert free >= missing + 2 * 1024**3, f"Need {(missing + 2 * 1024**3 - free) / 1024**3:.1f} GiB more free"
    print(f"MODEL_CACHE remaining_weight_download={missing / 1024**3:.2f} GiB", flush=True)
    for name, size, sha in SHARDS:
        path = MODEL_PATH / name
        if not verified(path, size, sha):
            blob = blobs / sha
            fetch_pinned(name, size, sha, blob)
            pending_link = path.with_name(path.name + ".verified-link")
            if pending_link.is_symlink():
                pending_link.unlink()
            pending_link.symlink_to(Path("../../blobs") / sha)
            pending_link.replace(path)
        print("MODEL_SHARD_VERIFIED", name, flush=True)
    for name, size, sha in METADATA:
        fetch_pinned(name, size, sha, MODEL_PATH / name)
    index = json.loads((MODEL_PATH / "model.safetensors.index.json").read_bytes())
    assert set(index["weight_map"].values()) == {name for name, _, _ in SHARDS}
    (WORK / "model_path.json").write_text(json.dumps(str(MODEL_PATH)), encoding="utf-8")
    print("MODEL_REVISION_PASS", MODEL_SHA, flush=True)


## 2. Screen media and compare the two refinement policies


In [7]:
ARTIFACT = 'HERMES_twinpast_refinement_evidence.zip'

if not VALIDATE_ONLY:
    (WORK/'launch.json').write_text(json.dumps(dict(started_utc=datetime.now(timezone.utc).isoformat(),
        source_revision=BASE_REVISION, model_revision=MODEL_SHA, transformers_revision=TF_SHA,
        embedded_sha256=EXPECTED, gpu=smi.stdout.strip()), indent=2), encoding='utf-8')
    env = {**ENV, 'PYTHONDONTWRITEBYTECODE': '1', 'HF_HUB_OFFLINE': '1'}
    logged([PYTHON, '-u', RUNNER, '--work', WORK, '--model', MODEL_PATH, '--media-reuse', MEDIA_REUSE, '--previous', PREVIOUS],
           'twinpast_refinement.log', env, REPO)
else:
    print('VALIDATION_PASS: setup and embedded sources checked without GPU, network or file writes')


MEDIA_READY videos/twinpast_0036/B.mp4 16.7 MiB
MEDIA_READY videos/twinpast_0001/A.mp4 12.6 MiB
MEDIA_READY videos/twinpast_0001/B.mp4 12.6 MiB
MEDIA_READY videos/twinpast_0037/A.mp4 10.4 MiB
MEDIA_READY videos/twinpast_0037/B.mp4 10.5 MiB
MEDIA_READY videos/twinpast_0153/A.mp4 7.8 MiB
MEDIA_READY videos/twinpast_0031/A.mp4 9.1 MiB
MEDIA_READY videos/twinpast_0153/B.mp4 7.9 MiB
MEDIA_READY videos/twinpast_0031/B.mp4 9.1 MiB
MEDIA_READY videos/twinpast_0291/A.mp4 10.5 MiB
MEDIA_READY videos/twinpast_0291/B.mp4 10.5 MiB
MEDIA_READY videos/twinpast_0083/A.mp4 19.2 MiB
MEDIA_READY videos/twinpast_0083/B.mp4 19.2 MiB
MEDIA_READY videos/twinpast_0132/A.mp4 11.6 MiB
MEDIA_READY videos/twinpast_0132/B.mp4 11.6 MiB
MEDIA_READY videos/twinpast_0326/A.mp4 37.1 MiB
MEDIA_READY videos/twinpast_0326/B.mp4 37.0 MiB
MEDIA_READY videos/twinpast_0255/A.mp4 15.9 MiB
MEDIA_READY videos/twinpast_0255/B.mp4 16.1 MiB
MEDIA_READY videos/twinpast_0118/A.mp4 15.0 MiB
MEDIA_READY videos/twinpast_0118/B.mp4 15.1 

## 3. Download the evidence ZIP

After a runner error, run this cell manually to retrieve failure evidence.


In [8]:

if not VALIDATE_ONLY:
    from IPython.display import FileLink, display
    artifact = WORK/ARTIFACT
    if artifact.is_file():
        link = Path.cwd()/ARTIFACT
        if artifact.resolve() != link.resolve():
            shutil.copyfile(artifact, link)
        print((WORK/'completion.json').read_text(encoding='utf-8'))
        display(FileLink(ARTIFACT))
    else:
        print('No runner ZIP yet. Inspect setup logs under', WORK/'logs')


{
  "status": "complete",
  "pairs": 8
}



/root/autodl-tmp/HERMES-reproduction/notebooks/autodl/HERMES_twinpast_refinement_evidence.zip